In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
FERRETTI ET AL. (2024) REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC
================================================================================

Complete reproduction with Digital Twin (3-D MLPA) integration + integrated AUC.
Now includes Neural Cox and Ensemble models!

DOMAINS:
- Clinical (C): 7 features (age, gender, T/N/M stage, overall stage, histology)
- Radiomics (R): 1706 features (from data3d_1706features)
- AutoEncoder (AE): 512 features (select top 15)
- Digital Twin (DT): 6 features from 3-D MLPA simulation
    * Sim_Growth_Rate   - Growth rate from MLPA cellular automaton simulation
    * Sim_Necrosis_Ratio - Ratio of necrotic cells from simulation
    * Bio_Alpha         - Proliferation rate α = 0.01 + 0.05 * normalized_entropy
    * Bio_Necrosis      - Necrosis rate β = 0.01 + 0.08 * (1 - sphericity)
    * Radiomics_Entropy - Tumor entropy (from histogram)
    * Radiomics_Sphericity - Tumor sphericity (surface area / volume)

MODELS:
- Cox Proportional Hazards (CoxPH): Classical survival model
- Neural Cox (DeepSurv-style): Neural network for non-linear survival prediction
- Ensemble: Weighted combination of CoxPH and Neural Cox predictions

METRICS:
- C-Index: Concordance index
- p-value: Log-rank test
- HR: Hazard ratio
- iAUC: Integrated time-dependent AUC

CONFIGURATIONS:
- Single domains: C, AE, R, DT
- Table 5 (Feature-Level Fusion): Concatenate features
- Table 6 (Signature-Level Fusion): Combine domain signatures
- Including DT combinations: DT+C, DT+R, DT+R+C, DT+AE+R+C

Reference: Ferretti et al. (2024) CMPB 258:108496
Author: Generated for NSCLC survival analysis with iAUC integration
================================================================================
"""

import numpy as np
import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Survival analysis
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

# Scikit-survival for integrated AUC and Gradient Boosting Survival
try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis
    from sksurv.linear_model import CoxnetSurvivalAnalysis
    SKSURV_AVAILABLE = True
except ImportError:
    print("⚠️  Warning: scikit-survival not installed. iAUC and some models will not be available.")
    print("   Install with: pip install scikit-survival")
    SKSURV_AVAILABLE = False

# PyTorch for Neural Cox
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    print("⚠️  Warning: PyTorch not installed. Neural Cox will use fallback.")
    print("   Install with: pip install torch")
    TORCH_AVAILABLE = False

# Machine learning
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
plt.style.use('default')

from tqdm import tqdm
import time

# =============================================================================
# CONFIGURATION
# =============================================================================

# ============== FILE PATHS ==============
# Radiomics features (1706 features)
DATA_DIR = Path('./data3d_1706features/outputs')
RADIOMICS_NPY_FILE = DATA_DIR / 'features_normalized_standard.npy'
RADIOMICS_NAMES_JSON = DATA_DIR / 'feature_names_normalized_standard.json'
PATIENT_IDS_JSON = DATA_DIR / 'patient_ids_normalized_standard.json'

# Clinical data
CLINICAL_FILE = Path('./nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv')

# AutoEncoder features (512 features)
DEEP_FEATURES_FILE = Path('./data_3d_ae_parallel/features/features_deep_512.csv')

# Digital Twin (3-D MLPA) features
DT_FEATURES_FILE = Path('./digital_twin_output/all_patients_biological_parameters.csv')

# Output directory
OUTPUT_DIR = Path('./ferretti_with_3d_mlpa_iauc_output')

# =============================================================================

# Digital Twin feature columns
DT_FEATURE_COLUMNS = [
    'Sim_Growth_Rate',
    'Sim_Necrosis_Ratio',
    'Bio_Alpha',
    'Bio_Necrosis',
    'Radiomics_Entropy',
    'Radiomics_Sphericity'
]

# Ferretti parameters
TOP_K_AE = 15
TOP_K_RADIOMICS = 10
CORRELATION_THRESHOLD = 0.70
LASSO_PENALIZER = 0.1
N_SPLITS = 5
RANDOM_STATE = 42

# Neural Cox parameters
NEURAL_COX_HIDDEN_LAYERS = [64, 32]  # Hidden layer sizes
NEURAL_COX_DROPOUT = 0.3
NEURAL_COX_EPOCHS = 100
NEURAL_COX_LR = 0.001
NEURAL_COX_BATCH_SIZE = 32
NEURAL_COX_PATIENCE = 10  # Early stopping patience

# Ensemble parameters
ENSEMBLE_WEIGHTS = {
    'coxph': 0.4,
    'neural_cox': 0.3,
    'gradient_boosting': 0.3
}

# Model selection: which models to run
RUN_COXPH = True
RUN_NEURAL_COX = True
RUN_ENSEMBLE = True

# Plotting
CREATE_KM_PLOTS = True
PLOT_DPI = 150

# iAUC evaluation time points (in years)
IAUC_TIME_POINTS = np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0])


# =============================================================================
# NEURAL COX MODEL (DeepSurv-style)
# =============================================================================

if TORCH_AVAILABLE:
    class NeuralCoxModel(nn.Module):
        """
        Neural Cox Proportional Hazards Model (DeepSurv-style).
        
        This network outputs a risk score that is used in the Cox partial likelihood.
        Higher risk score = worse prognosis (higher hazard).
        """
        
        def __init__(self, input_dim: int, hidden_layers: List[int] = [64, 32], 
                     dropout: float = 0.3):
            super(NeuralCoxModel, self).__init__()
            
            layers = []
            prev_dim = input_dim
            
            for hidden_dim in hidden_layers:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                prev_dim = hidden_dim
            
            # Final output layer (single risk score)
            layers.append(nn.Linear(prev_dim, 1))
            
            self.network = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.network(x)
    
    
    def cox_partial_likelihood_loss(risk_scores: torch.Tensor, 
                                    times: torch.Tensor, 
                                    events: torch.Tensor) -> torch.Tensor:
        """
        Negative log partial likelihood for Cox model.
        
        Parameters:
        -----------
        risk_scores : tensor of shape (batch_size, 1)
            Predicted risk scores from the network
        times : tensor of shape (batch_size,)
            Survival times
        events : tensor of shape (batch_size,)
            Event indicators (1=event, 0=censored)
        
        Returns:
        --------
        loss : scalar tensor
            Negative log partial likelihood
        """
        risk_scores = risk_scores.squeeze()
        
        # Sort by time (descending for risk set computation)
        sorted_indices = torch.argsort(times, descending=True)
        sorted_risk = risk_scores[sorted_indices]
        sorted_events = events[sorted_indices]
        
        # Compute log risk for each subject
        max_risk = sorted_risk.max()
        log_risk = sorted_risk - max_risk
        
        # Cumulative sum of exp(risk) for risk sets (from longest to shortest survival)
        exp_risk = torch.exp(log_risk)
        cumsum_exp_risk = torch.cumsum(exp_risk, dim=0)
        
        # Log of cumulative sum
        log_cumsum = torch.log(cumsum_exp_risk + 1e-7) + max_risk
        
        # Partial likelihood: sum over events
        # L = sum_{i: event} [ risk_i - log(sum_{j in R_i} exp(risk_j)) ]
        event_mask = sorted_events == 1
        
        if event_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True)
        
        partial_likelihood = (sorted_risk[event_mask] - log_cumsum[event_mask]).sum()
        
        # Return negative (for minimization)
        return -partial_likelihood / event_mask.sum()
    
    
    class NeuralCoxTrainer:
        """Trainer for Neural Cox model with early stopping."""
        
        def __init__(self, input_dim: int, 
                     hidden_layers: List[int] = [64, 32],
                     dropout: float = 0.3,
                     lr: float = 0.001,
                     epochs: int = 100,
                     batch_size: int = 32,
                     patience: int = 10,
                     device: str = None):
            
            self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
            self.model = NeuralCoxModel(input_dim, hidden_layers, dropout).to(self.device)
            self.optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)
            self.epochs = epochs
            self.batch_size = batch_size
            self.patience = patience
            self.best_loss = float('inf')
            self.best_state = None
            self.patience_counter = 0
        
        def fit(self, X_train: np.ndarray, time_train: np.ndarray, event_train: np.ndarray,
                X_val: np.ndarray = None, time_val: np.ndarray = None, event_val: np.ndarray = None):
            """Train the Neural Cox model."""
            
            # Convert to tensors
            X_tensor = torch.FloatTensor(X_train).to(self.device)
            time_tensor = torch.FloatTensor(time_train).to(self.device)
            event_tensor = torch.FloatTensor(event_train).to(self.device)
            
            # Create dataset
            dataset = TensorDataset(X_tensor, time_tensor, event_tensor)
            dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
            
            # Validation data
            if X_val is not None:
                X_val_tensor = torch.FloatTensor(X_val).to(self.device)
                time_val_tensor = torch.FloatTensor(time_val).to(self.device)
                event_val_tensor = torch.FloatTensor(event_val).to(self.device)
            
            self.model.train()
            
            for epoch in range(self.epochs):
                epoch_loss = 0.0
                n_batches = 0
                
                for batch_X, batch_time, batch_event in dataloader:
                    self.optimizer.zero_grad()
                    
                    risk_scores = self.model(batch_X)
                    loss = cox_partial_likelihood_loss(risk_scores, batch_time, batch_event)
                    
                    if torch.isnan(loss) or torch.isinf(loss):
                        continue
                    
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()
                    
                    epoch_loss += loss.item()
                    n_batches += 1
                
                # Validation and early stopping
                if X_val is not None and n_batches > 0:
                    self.model.eval()
                    with torch.no_grad():
                        val_risk = self.model(X_val_tensor)
                        val_loss = cox_partial_likelihood_loss(val_risk, time_val_tensor, event_val_tensor)
                    
                    if val_loss < self.best_loss:
                        self.best_loss = val_loss
                        self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                        self.patience_counter = 0
                    else:
                        self.patience_counter += 1
                    
                    if self.patience_counter >= self.patience:
                        break
                    
                    self.model.train()
            
            # Load best model
            if self.best_state is not None:
                self.model.load_state_dict(self.best_state)
        
        def predict_risk(self, X: np.ndarray) -> np.ndarray:
            """Predict risk scores for new data."""
            self.model.eval()
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X).to(self.device)
                risk_scores = self.model(X_tensor).cpu().numpy().flatten()
            return risk_scores


# =============================================================================
# GRADIENT BOOSTING SURVIVAL MODEL (from scikit-survival)
# =============================================================================

class GradientBoostingSurvival:
    """Wrapper for scikit-survival's Gradient Boosting Survival Analysis."""
    
    def __init__(self, n_estimators: int = 100, learning_rate: float = 0.1,
                 max_depth: int = 3, min_samples_split: int = 10,
                 random_state: int = 42):
        
        if not SKSURV_AVAILABLE:
            raise ImportError("scikit-survival is required for GradientBoostingSurvival")
        
        self.model = GradientBoostingSurvivalAnalysis(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=random_state,
            subsample=0.8
        )
    
    def fit(self, X: np.ndarray, time: np.ndarray, event: np.ndarray):
        """Fit the gradient boosting model."""
        # Create structured array for scikit-survival
        y = np.array([(bool(e), t) for e, t in zip(event, time)],
                     dtype=[('event', bool), ('time', float)])
        self.model.fit(X, y)
        return self
    
    def predict_risk(self, X: np.ndarray) -> np.ndarray:
        """Predict risk scores (higher = worse prognosis)."""
        return self.model.predict(X)


# =============================================================================
# ENSEMBLE MODEL
# =============================================================================

class EnsembleSurvivalModel:
    """
    Ensemble model combining CoxPH, Neural Cox, and Gradient Boosting.
    
    The ensemble averages normalized risk scores from multiple models
    using specified weights.
    """
    
    def __init__(self, weights: Dict[str, float] = None, use_neural_cox: bool = True,
                 use_gradient_boosting: bool = True):
        
        self.weights = weights or {'coxph': 0.4, 'neural_cox': 0.3, 'gradient_boosting': 0.3}
        self.use_neural_cox = use_neural_cox and TORCH_AVAILABLE
        self.use_gradient_boosting = use_gradient_boosting and SKSURV_AVAILABLE
        
        # Normalize weights based on available models
        self._normalize_weights()
        
        self.coxph_model = None
        self.neural_cox_trainer = None
        self.gb_model = None
        self.scaler = StandardScaler()
    
    def _normalize_weights(self):
        """Normalize weights to sum to 1 based on available models."""
        active_weights = {'coxph': self.weights.get('coxph', 0.4)}
        
        if self.use_neural_cox:
            active_weights['neural_cox'] = self.weights.get('neural_cox', 0.3)
        if self.use_gradient_boosting:
            active_weights['gradient_boosting'] = self.weights.get('gradient_boosting', 0.3)
        
        total = sum(active_weights.values())
        self.active_weights = {k: v/total for k, v in active_weights.items()}
    
    def fit(self, X_train: np.ndarray, time_train: np.ndarray, event_train: np.ndarray,
            feature_names: List[str] = None):
        """Fit all component models."""
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X_train)
        
        # 1. Fit CoxPH
        try:
            df_train = pd.DataFrame(X_scaled, columns=feature_names or [f'F{i}' for i in range(X_scaled.shape[1])])
            df_train['time_years'] = time_train
            df_train['event'] = event_train
            
            self.coxph_model = CoxPHFitter(penalizer=LASSO_PENALIZER)
            feature_cols = [c for c in df_train.columns if c not in ['time_years', 'event']]
            self.coxph_model.fit(df_train[feature_cols + ['time_years', 'event']],
                                 duration_col='time_years', event_col='event')
            self.coxph_features = feature_cols
        except Exception as e:
            print(f"   ⚠️  CoxPH fitting failed: {str(e)[:50]}")
            self.coxph_model = None
        
        # 2. Fit Neural Cox
        if self.use_neural_cox:
            try:
                self.neural_cox_trainer = NeuralCoxTrainer(
                    input_dim=X_scaled.shape[1],
                    hidden_layers=NEURAL_COX_HIDDEN_LAYERS,
                    dropout=NEURAL_COX_DROPOUT,
                    lr=NEURAL_COX_LR,
                    epochs=NEURAL_COX_EPOCHS,
                    batch_size=NEURAL_COX_BATCH_SIZE,
                    patience=NEURAL_COX_PATIENCE
                )
                
                # Use 20% of training data for validation within Neural Cox
                n_val = max(1, int(len(X_scaled) * 0.2))
                indices = np.random.permutation(len(X_scaled))
                val_idx, train_idx = indices[:n_val], indices[n_val:]
                
                self.neural_cox_trainer.fit(
                    X_scaled[train_idx], time_train[train_idx], event_train[train_idx],
                    X_scaled[val_idx], time_train[val_idx], event_train[val_idx]
                )
            except Exception as e:
                print(f"   ⚠️  Neural Cox fitting failed: {str(e)[:50]}")
                self.neural_cox_trainer = None
        
        # 3. Fit Gradient Boosting
        if self.use_gradient_boosting:
            try:
                self.gb_model = GradientBoostingSurvival(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    random_state=RANDOM_STATE
                )
                self.gb_model.fit(X_scaled, time_train, event_train)
            except Exception as e:
                print(f"   ⚠️  Gradient Boosting fitting failed: {str(e)[:50]}")
                self.gb_model = None
        
        return self
    
    def predict_risk(self, X: np.ndarray) -> np.ndarray:
        """Predict ensemble risk scores."""
        
        X_scaled = self.scaler.transform(X)
        
        risk_scores = []
        weights = []
        
        # CoxPH predictions
        if self.coxph_model is not None:
            try:
                df_pred = pd.DataFrame(X_scaled, columns=self.coxph_features)
                coxph_risk = self.coxph_model.predict_partial_hazard(df_pred).values.flatten()
                # Normalize to [0, 1]
                coxph_risk = (coxph_risk - coxph_risk.min()) / (coxph_risk.max() - coxph_risk.min() + 1e-8)
                risk_scores.append(coxph_risk)
                weights.append(self.active_weights.get('coxph', 0))
            except:
                pass
        
        # Neural Cox predictions
        if self.neural_cox_trainer is not None:
            try:
                neural_risk = self.neural_cox_trainer.predict_risk(X_scaled)
                # Normalize to [0, 1]
                neural_risk = (neural_risk - neural_risk.min()) / (neural_risk.max() - neural_risk.min() + 1e-8)
                risk_scores.append(neural_risk)
                weights.append(self.active_weights.get('neural_cox', 0))
            except:
                pass
        
        # Gradient Boosting predictions
        if self.gb_model is not None:
            try:
                gb_risk = self.gb_model.predict_risk(X_scaled)
                # Normalize to [0, 1]
                gb_risk = (gb_risk - gb_risk.min()) / (gb_risk.max() - gb_risk.min() + 1e-8)
                risk_scores.append(gb_risk)
                weights.append(self.active_weights.get('gradient_boosting', 0))
            except:
                pass
        
        if len(risk_scores) == 0:
            return np.zeros(len(X))
        
        # Weighted average
        weights = np.array(weights)
        weights = weights / weights.sum()  # Renormalize
        
        ensemble_risk = np.zeros(len(X))
        for risk, w in zip(risk_scores, weights):
            ensemble_risk += w * risk
        
        return ensemble_risk


# =============================================================================
# DATA LOADING
# =============================================================================

def load_clinical_data(clinical_file: Path) -> pd.DataFrame:
    """Load clinical data with 7 features (Ferretti method)."""
    
    print("\n" + "="*100)
    print("📊 LOADING CLINICAL DATA")
    print("="*100)
    
    df = pd.read_csv(clinical_file)
    df.columns = [c.strip() for c in df.columns]
    
    print(f"\n✅ Loaded {len(df)} patients")
    
    # Standardize column names
    col_map = {}
    for col in df.columns:
        col_lower = col.lower()
        if 'patient' in col_lower:
            col_map[col] = 'PatientID'
        elif col_lower == 'age':
            col_map[col] = 'age'
        elif 't.stage' in col_lower or 't stage' in col_lower:
            col_map[col] = 'T_Stage'
        elif 'n.stage' in col_lower or 'n stage' in col_lower:
            col_map[col] = 'N_Stage'
        elif 'm.stage' in col_lower or 'm stage' in col_lower:
            col_map[col] = 'M_Stage'
        elif 'overall' in col_lower and 'stage' in col_lower:
            col_map[col] = 'Overall_Stage'
        elif 'histology' in col_lower:
            col_map[col] = 'Histology'
        elif 'gender' in col_lower or 'sex' in col_lower:
            col_map[col] = 'gender'
        elif 'survival' in col_lower and 'time' in col_lower:
            col_map[col] = 'Survival_time'
        elif 'dead' in col_lower or 'event' in col_lower or 'status' in col_lower:
            col_map[col] = 'event'
    
    df = df.rename(columns=col_map)
    df['PatientID'] = df['PatientID'].astype(str)
    
    # Survival time
    df['Survival_time'] = pd.to_numeric(df['Survival_time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce')
    df['time_years'] = df['Survival_time'] / 365.25
    
    print(f"   Time range: {df['time_years'].min():.2f} - {df['time_years'].max():.2f} years")
    print(f"   Events: {df['event'].sum():.0f}/{len(df)} ({100*df['event'].mean():.1f}%)")
    
    # Missing data imputation
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age'].fillna(df['age'].median(), inplace=True)
    
    for stage_col in ['T_Stage', 'N_Stage', 'M_Stage']:
        if stage_col in df.columns:
            df[stage_col] = pd.to_numeric(df[stage_col], errors='coerce')
            mode_val = df[stage_col].mode()[0] if len(df[stage_col].mode()) > 0 else 0
            df[stage_col].fillna(mode_val, inplace=True)
        else:
            df[stage_col] = 0
    
    if 'Overall_Stage' in df.columns:
        mode_val = df['Overall_Stage'].mode()[0] if len(df['Overall_Stage'].mode()) > 0 else 'IIIb'
        df['Overall_Stage'].fillna(mode_val, inplace=True)
    else:
        df['Overall_Stage'] = 'IIIb'
    
    if 'Histology' in df.columns:
        mode_val = df['Histology'].mode()[0] if len(df['Histology'].mode()) > 0 else 'nos'
        df['Histology'].fillna(mode_val, inplace=True)
    else:
        df['Histology'] = 'nos'
    
    if 'gender' in df.columns:
        mode_val = df['gender'].mode()[0] if len(df['gender'].mode()) > 0 else 'male'
        df['gender'].fillna(mode_val, inplace=True)
    else:
        df['gender'] = 'male'
    
    # Encode categorical variables
    df['gender_encoded'] = df['gender'].astype(str).str.lower().str.strip().map({
        'male': 1, 'female': 0, 'm': 1, 'f': 0
    }).fillna(1)
    
    df['T_Stage_encoded'] = pd.to_numeric(df['T_Stage'], errors='coerce').fillna(0)
    df['N_Stage_encoded'] = pd.to_numeric(df['N_Stage'], errors='coerce').fillna(0)
    df['M_Stage_encoded'] = pd.to_numeric(df['M_Stage'], errors='coerce').fillna(0)
    
    stage_map = {
        'i': 1, 'ia': 1, 'ib': 1,
        'ii': 2, 'iia': 2, 'iib': 2,
        'iii': 3, 'iiia': 3, 'iiib': 4,
        'iv': 5, 'iva': 5, 'ivb': 5
    }
    df['Overall_Stage_encoded'] = df['Overall_Stage'].astype(str).str.lower().str.strip().map(stage_map).fillna(0)
    
    histology_map = {}
    for val in df['Histology'].unique():
        val_str = str(val).lower().strip()
        if 'large' in val_str or 'lcc' in val_str:
            histology_map[val] = 0
        elif 'squamous' in val_str or 'scc' in val_str:
            histology_map[val] = 1
        elif 'adeno' in val_str:
            histology_map[val] = 2
        else:
            histology_map[val] = 3
    
    df['Histology_encoded'] = df['Histology'].map(histology_map).fillna(3)
    
    df = df.dropna(subset=['time_years', 'event'])
    
    print(f"\n✅ Clinical data prepared: {len(df)} patients with 7 features")
    
    return df


def load_radiomics_from_npy(npy_file: Path, 
                           names_file: Path,
                           ids_file: Path) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """Load radiomics features from .npy files (1706 features)."""
    
    print("\n" + "="*100)
    print("📊 LOADING RADIOMICS FEATURES (1706 features)")
    print("="*100)
    
    print(f"\n📂 Loading: {npy_file}")
    if not npy_file.exists():
        raise FileNotFoundError(f"Radiomics file not found: {npy_file}")
    
    radiomics_array = np.load(npy_file)
    print(f"   ✅ Shape: {radiomics_array.shape}")
    
    with open(names_file, 'r') as f:
        feature_names = json.load(f)
    
    with open(ids_file, 'r') as f:
        patient_ids = json.load(f)
    
    df_radiomics = pd.DataFrame(radiomics_array, columns=feature_names)
    df_radiomics['PatientID'] = patient_ids
    
    cols = ['PatientID'] + [c for c in df_radiomics.columns if c != 'PatientID']
    df_radiomics = df_radiomics[cols]
    
    print(f"   ✅ Patients: {len(patient_ids)}, Features: {len(feature_names)}")
    
    return df_radiomics, feature_names, patient_ids


def load_ae_features(ae_file: Path) -> Tuple[pd.DataFrame, List[str]]:
    """Load AutoEncoder features (512 features)."""
    
    print("\n" + "="*100)
    print("📊 LOADING AUTOENCODER FEATURES (512 features)")
    print("="*100)
    
    print(f"\n📂 Loading: {ae_file}")
    df_ae = pd.read_csv(ae_file)
    
    ae_features = [c for c in df_ae.columns if 'Deep_F' in c or 'deep_f' in c.lower()]
    
    print(f"   ✅ Patients: {len(df_ae)}, Features: {len(ae_features)}")
    
    return df_ae, ae_features


def load_digital_twin_features(dt_file: Path) -> Tuple[pd.DataFrame, List[str]]:
    """Load Digital Twin (3-D MLPA) features."""
    
    print("\n" + "="*100)
    print("📊 LOADING DIGITAL TWIN (3-D MLPA) FEATURES")
    print("="*100)
    
    print(f"\n📂 Loading: {dt_file}")
    
    if not dt_file.exists():
        print(f"   ⚠️  Digital Twin file not found: {dt_file}")
        print(f"   Please run digital_twin_pipeline.py first to generate features.")
        return pd.DataFrame({'PatientID': []}), []
    
    df_dt = pd.read_csv(dt_file)
    df_dt['PatientID'] = df_dt['PatientID'].astype(str)
    
    available_dt_features = [f for f in DT_FEATURE_COLUMNS if f in df_dt.columns]
    missing_features = [f for f in DT_FEATURE_COLUMNS if f not in df_dt.columns]
    
    if missing_features:
        print(f"   ⚠️  Missing features: {missing_features}")
    
    print(f"   ✅ Patients: {len(df_dt)}")
    print(f"   ✅ Available DT features: {len(available_dt_features)}")
    
    print(f"\n   📊 3-D MLPA Feature Statistics:")
    print(f"   {'─'*70}")
    print(f"   {'Feature':<25} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
    print(f"   {'─'*70}")
    
    for feat in available_dt_features:
        values = df_dt[feat]
        print(f"   {feat:<25} {values.mean():>12.6f} {values.std():>12.6f} {values.min():>12.6f} {values.max():>12.6f}")
    
    print(f"   {'─'*70}")
    
    return df_dt, available_dt_features


# =============================================================================
# FEATURE SELECTION (FERRETTI PIPELINE)
# =============================================================================

def remove_correlated_features(df: pd.DataFrame, 
                               features: List[str], 
                               threshold: float = 0.70) -> List[str]:
    """Remove highly correlated features (Ferretti method)."""
    
    if len(features) <= 1:
        return features
    
    try:
        correlation_matrix = df[features].corr().abs()
        features_to_remove = set()
        
        for i in range(len(correlation_matrix.columns)):
            for j in range(i + 1, len(correlation_matrix.columns)):
                if correlation_matrix.iloc[i, j] >= threshold:
                    avg_corr_i = correlation_matrix.iloc[i, :].mean()
                    avg_corr_j = correlation_matrix.iloc[j, :].mean()
                    
                    if avg_corr_i > avg_corr_j:
                        features_to_remove.add(correlation_matrix.columns[i])
                    else:
                        features_to_remove.add(correlation_matrix.columns[j])
        
        selected_features = [f for f in features if f not in features_to_remove]
        return selected_features
        
    except Exception as e:
        print(f"   ⚠️  Error in correlation filtering: {str(e)[:100]}")
        return features


def remove_constant_features(df: pd.DataFrame, features: List[str]) -> List[str]:
    """Remove features with zero variance."""
    
    if len(features) == 0:
        return features
    
    try:
        variances = df[features].var()
        non_constant = variances[variances > 0].index.tolist()
        return non_constant
    except:
        return features


def select_features_univariate_cox(train_df: pd.DataFrame, 
                                   features: List[str], 
                                   top_k: int) -> List[str]:
    """Select top K features using univariate Cox PH (by p-value)."""
    
    if len(features) <= top_k:
        return features
    
    p_values = []
    
    for f in features:
        try:
            cph = CoxPHFitter()
            subset = train_df[[f, 'time_years', 'event']].copy()
            cph.fit(subset, duration_col='time_years', event_col='event')
            p_values.append((f, cph.summary.loc[f, 'p']))
        except:
            p_values.append((f, 1.0))
    
    p_values.sort(key=lambda x: x[1])
    selected = [x[0] for x in p_values[:top_k]]
    
    return selected


# =============================================================================
# SIGNATURE GENERATION (TABLE 6 APPROACH)
# =============================================================================

def generate_domain_signature(train_df: pd.DataFrame, 
                              val_df: pd.DataFrame,
                              features: List[str], 
                              domain_name: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Generate domain-specific signature (risk scores)."""
    
    if len(features) == 0:
        return None, None
    
    try:
        scaler = StandardScaler()
        train_scaled = train_df.copy()
        val_scaled = val_df.copy()
        train_scaled[features] = scaler.fit_transform(train_df[features])
        val_scaled[features] = scaler.transform(val_df[features])
        
        cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
        cph.fit(train_scaled[features + ['time_years', 'event']],
               duration_col='time_years', event_col='event')
        
        train_signature = cph.predict_partial_hazard(train_scaled[features]).values.flatten()
        val_signature = cph.predict_partial_hazard(val_scaled[features]).values.flatten()
        
        return train_signature, val_signature
        
    except Exception as e:
        print(f"   ⚠️  Error generating {domain_name} signature: {str(e)[:50]}")
        return None, None


# =============================================================================
# SURVIVAL METRICS WITH INTEGRATED AUC
# =============================================================================

def calculate_all_metrics(time_test: np.ndarray, 
                         event_test: np.ndarray,
                         risk_scores: np.ndarray,
                         time_train: np.ndarray = None,
                         event_train: np.ndarray = None) -> Dict[str, float]:
    """
    Calculate C-Index, log-rank p-value, Hazard Ratio, and integrated AUC.
    
    Parameters:
    -----------
    time_test : array-like
        Survival times for test set (in years)
    event_test : array-like
        Event indicators for test set (1=event, 0=censored)
    risk_scores : array-like
        Predicted risk scores (higher = higher risk)
    time_train : array-like, optional
        Survival times for training set (needed for iAUC)
    event_train : array-like, optional
        Event indicators for training set (needed for iAUC)
    
    Returns:
    --------
    metrics : dict
        Dictionary containing:
        - c_index: Concordance index
        - p_value: Log-rank test p-value
        - hazard_ratio: Hazard ratio (high vs low risk)
        - iauc: Integrated time-dependent AUC
        - auc_by_time: AUC at individual time points
    """
    
    metrics = {}
    
    # 1. C-Index (Concordance Index)
    try:
        c_index = concordance_index(time_test, -risk_scores, event_test)
        metrics['c_index'] = c_index
    except:
        metrics['c_index'] = np.nan
    
    # 2. Log-rank p-value (stratified by median risk)
    try:
        median_risk = np.median(risk_scores)
        group = (risk_scores >= median_risk).astype(int)
        
        results = logrank_test(
            time_test[group == 0], time_test[group == 1],
            event_test[group == 0], event_test[group == 1]
        )
        metrics['p_value'] = results.p_value
    except:
        metrics['p_value'] = np.nan
    
    # 3. Hazard Ratio (high risk vs low risk)
    try:
        median_risk = np.median(risk_scores)
        group = (risk_scores >= median_risk).astype(int)
        
        df_temp = pd.DataFrame({
            'time': time_test,
            'event': event_test,
            'risk_group': group
        })
        
        cph = CoxPHFitter()
        cph.fit(df_temp, duration_col='time', event_col='event')
        hr = np.exp(cph.params_['risk_group'])
        metrics['hazard_ratio'] = hr
    except:
        metrics['hazard_ratio'] = np.nan
    
    # 4. Integrated AUC (iAUC) - requires scikit-survival
    if SKSURV_AVAILABLE and time_train is not None and event_train is not None:
        try:
            # Create structured arrays for scikit-survival
            train_y = np.array(
                [(bool(e), t) for e, t in zip(event_train, time_train)],
                dtype=[('event', bool), ('time', float)]
            )
            
            test_y = np.array(
                [(bool(e), t) for e, t in zip(event_test, time_test)],
                dtype=[('event', bool), ('time', float)]
            )
            
            # Define evaluation time points
            # Use times where there are enough events
            max_time = min(np.percentile(time_test[event_test == 1], 90), 2.0)
            times = IAUC_TIME_POINTS[IAUC_TIME_POINTS <= max_time]
            
            # Filter to ensure we have events at these times
            min_time = max(0.1, np.percentile(time_test[event_test == 1], 10))
            times = times[times >= min_time]
            
            if len(times) >= 2:
                # Calculate cumulative/dynamic AUC at multiple time points
                auc_scores, mean_auc = cumulative_dynamic_auc(
                    train_y, 
                    test_y, 
                    risk_scores,  # Higher risk score = worse prognosis
                    times
                )
                
                # Integrated AUC is the mean of time-dependent AUCs
                metrics['iauc'] = mean_auc
                
                # Store individual time-point AUCs for detailed analysis
                metrics['auc_by_time'] = dict(zip(times, auc_scores))
                
                # Also store min/max for analysis
                metrics['iauc_min'] = np.min(auc_scores)
                metrics['iauc_max'] = np.max(auc_scores)
                
            else:
                metrics['iauc'] = np.nan
                metrics['auc_by_time'] = {}
                metrics['iauc_min'] = np.nan
                metrics['iauc_max'] = np.nan
                
        except Exception as e:
            print(f"   ⚠️  Error calculating iAUC: {str(e)[:100]}")
            metrics['iauc'] = np.nan
            metrics['auc_by_time'] = {}
            metrics['iauc_min'] = np.nan
            metrics['iauc_max'] = np.nan
    else:
        metrics['iauc'] = np.nan
        metrics['auc_by_time'] = {}
        metrics['iauc_min'] = np.nan
        metrics['iauc_max'] = np.nan
        
        if not SKSURV_AVAILABLE and time_train is not None:
            # Only warn once
            pass
    
    return metrics


# =============================================================================
# KAPLAN-MEIER PLOTTING
# =============================================================================

def plot_km_curve(time: np.ndarray, 
                  event: np.ndarray, 
                  risk_scores: np.ndarray,
                  title: str, 
                  output_path: Path, 
                  dataset_type: str = "Validation"):
    """Plot Kaplan-Meier curve (high risk vs low risk)."""
    
    try:
        median_risk = np.median(risk_scores)
        high_risk_mask = risk_scores >= median_risk
        
        kmf_high = KaplanMeierFitter()
        kmf_low = KaplanMeierFitter()
        
        kmf_high.fit(time[high_risk_mask], event[high_risk_mask], label='High Risk')
        kmf_low.fit(time[~high_risk_mask], event[~high_risk_mask], label='Low Risk')
        
        results = logrank_test(
            time[high_risk_mask], time[~high_risk_mask],
            event[high_risk_mask], event[~high_risk_mask]
        )
        
        fig, ax = plt.subplots(figsize=(10, 7))
        
        kmf_high.plot_survival_function(ax=ax, ci_show=True, color='red', linewidth=2.5, alpha=0.8)
        kmf_low.plot_survival_function(ax=ax, ci_show=True, color='blue', linewidth=2.5, alpha=0.8)
        
        significance = '✅ Significant' if results.p_value < 0.05 else '❌ Not Significant'
        ax.set_title(f'{title} ({dataset_type})\nLog-Rank p = {results.p_value:.6f} ({significance})',
                    fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Years', fontsize=12, fontweight='bold')
        ax.set_ylabel('Survival Probability', fontsize=12, fontweight='bold')
        ax.legend(fontsize=11, loc='best', framealpha=0.9)
        ax.grid(alpha=0.3, linestyle='--')
        ax.set_ylim([0, 1.05])
        
        n_high = high_risk_mask.sum()
        n_low = (~high_risk_mask).sum()
        events_high = event[high_risk_mask].sum()
        events_low = event[~high_risk_mask].sum()
        
        stats_text = (f'High Risk: n={n_high}, events={int(events_high)}\n'
                     f'Low Risk: n={n_low}, events={int(events_low)}')
        
        ax.text(0.02, 0.02, stats_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight')
        plt.close()
        
        return results.p_value
        
    except Exception as e:
        print(f"   ⚠️  Error plotting KM curve: {str(e)[:50]}")
        return np.nan


# =============================================================================
# TRAIN MODEL HELPER FUNCTION
# =============================================================================

def train_and_predict(model_type: str,
                      fold_train: pd.DataFrame,
                      fold_val: pd.DataFrame,
                      fold_features: List[str]) -> Optional[np.ndarray]:
    """
    Train a model and return predictions.
    
    Parameters:
    -----------
    model_type : str
        One of 'coxph', 'neural_cox', 'ensemble'
    fold_train : DataFrame
        Training data with features + time_years + event
    fold_val : DataFrame
        Validation data
    fold_features : list
        List of feature column names
    
    Returns:
    --------
    risk_scores : array or None
        Predicted risk scores for validation set
    """
    
    scaler = StandardScaler()
    X_train = fold_train[fold_features].values
    X_val = fold_val[fold_features].values
    time_train = fold_train['time_years'].values
    event_train = fold_train['event'].values
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # =========================
    # CoxPH Model
    # =========================
    if model_type == 'coxph':
        try:
            fold_train_scaled = fold_train.copy()
            fold_val_scaled = fold_val.copy()
            fold_train_scaled[fold_features] = X_train_scaled
            fold_val_scaled[fold_features] = X_val_scaled
            
            cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
            cph.fit(fold_train_scaled[fold_features + ['time_years', 'event']],
                   duration_col='time_years', event_col='event')
            
            risk_scores_val = cph.predict_partial_hazard(fold_val_scaled[fold_features]).values.flatten()
            return risk_scores_val
        except:
            return None
    
    # =========================
    # Neural Cox Model
    # =========================
    elif model_type == 'neural_cox':
        if not TORCH_AVAILABLE:
            return None
        
        try:
            trainer = NeuralCoxTrainer(
                input_dim=X_train_scaled.shape[1],
                hidden_layers=NEURAL_COX_HIDDEN_LAYERS,
                dropout=NEURAL_COX_DROPOUT,
                lr=NEURAL_COX_LR,
                epochs=NEURAL_COX_EPOCHS,
                batch_size=NEURAL_COX_BATCH_SIZE,
                patience=NEURAL_COX_PATIENCE
            )
            
            # Split training data for internal validation
            n_internal_val = max(1, int(len(X_train_scaled) * 0.15))
            indices = np.random.permutation(len(X_train_scaled))
            internal_val_idx, internal_train_idx = indices[:n_internal_val], indices[n_internal_val:]
            
            trainer.fit(
                X_train_scaled[internal_train_idx],
                time_train[internal_train_idx],
                event_train[internal_train_idx],
                X_train_scaled[internal_val_idx],
                time_train[internal_val_idx],
                event_train[internal_val_idx]
            )
            
            risk_scores_val = trainer.predict_risk(X_val_scaled)
            return risk_scores_val
        except Exception as e:
            print(f"      Neural Cox error: {str(e)[:50]}")
            return None
    
    # =========================
    # Ensemble Model
    # =========================
    elif model_type == 'ensemble':
        try:
            ensemble = EnsembleSurvivalModel(
                weights=ENSEMBLE_WEIGHTS,
                use_neural_cox=TORCH_AVAILABLE,
                use_gradient_boosting=SKSURV_AVAILABLE
            )
            
            ensemble.fit(X_train_scaled, time_train, event_train, 
                        feature_names=fold_features)
            
            risk_scores_val = ensemble.predict_risk(X_val_scaled)
            return risk_scores_val
        except Exception as e:
            print(f"      Ensemble error: {str(e)[:50]}")
            return None
    
    return None


# =============================================================================
# MAIN EXPERIMENT WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS
# =============================================================================

def run_experiment_with_3d_mlpa(df_merged: pd.DataFrame,
                                radiomics_features: List[str],
                                ae_features: List[str],
                                clinical_features: List[str],
                                dt_features: List[str],
                                n_splits: int = 5,
                                random_state: int = 42,
                                create_km_plots: bool = True) -> Dict:
    """
    Run experiment with 3-D MLPA Digital Twin features, integrated AUC,
    and multiple models (CoxPH, Neural Cox, Ensemble).
    
    Configurations:
    - Single domains: C, AE, R, DT
    - Table 5 (Feature-level fusion)
    - Table 6 (Signature-level fusion)
    - Including DT combinations
    
    Each configuration is tested with CoxPH, Neural Cox, and Ensemble models.
    """
    
    print("\n" + "="*100)
    print("📊 RUNNING EXPERIMENT WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS")
    print("="*100)
    
    print("\n🔧 MODEL AVAILABILITY:")
    print(f"   • CoxPH:            ✅ Available (lifelines)")
    print(f"   • Neural Cox:       {'✅ Available (PyTorch)' if TORCH_AVAILABLE else '❌ Not available (install PyTorch)'}")
    print(f"   • Gradient Boosting: {'✅ Available (scikit-survival)' if SKSURV_AVAILABLE else '❌ Not available'}")
    print(f"   • Ensemble:         {'✅ Available' if (TORCH_AVAILABLE or SKSURV_AVAILABLE) else '⚠️ Limited (CoxPH only)'}")
    print(f"   • iAUC:             {'✅ Available' if SKSURV_AVAILABLE else '❌ Not available'}")
    
    # Determine which models to run
    models_to_run = []
    if RUN_COXPH:
        models_to_run.append('coxph')
    if RUN_NEURAL_COX and TORCH_AVAILABLE:
        models_to_run.append('neural_cox')
    if RUN_ENSEMBLE:
        models_to_run.append('ensemble')
    
    print(f"\n🔬 MODELS TO RUN: {models_to_run}")
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Check if DT features are available
    has_dt = len(dt_features) > 0
    
    if not has_dt:
        print("\n⚠️  No Digital Twin features available. Skipping DT configurations.")
    else:
        print(f"\n✅ Digital Twin features available ({len(dt_features)})")
    
    # Define all configurations (feature combinations)
    configs = {}
    
    # ========== SINGLE DOMAINS ==========
    configs['C_Only'] = {
        'description': f'Clinical only ({len(clinical_features)} features)',
        'fusion_type': None,
        'features': clinical_features,
        'use_selection': False
    }
    
    configs['AE_Only'] = {
        'description': f'AutoEncoder (top {TOP_K_AE} from {len(ae_features)})',
        'fusion_type': None,
        'features': ae_features,
        'use_selection': True,
        'top_k': TOP_K_AE
    }
    
    configs['R_Only'] = {
        'description': f'Radiomics (top {TOP_K_RADIOMICS} from {len(radiomics_features)})',
        'fusion_type': None,
        'features': radiomics_features,
        'use_selection': True,
        'top_k': TOP_K_RADIOMICS
    }
    
    if has_dt:
        configs['DT_Only'] = {
            'description': f'Digital Twin 3-D MLPA ({len(dt_features)} features)',
            'fusion_type': None,
            'features': dt_features,
            'use_selection': False
        }
    
    # ========== TABLE 5: FEATURE-LEVEL FUSION ==========
    configs['Table5_AE_R'] = {
        'description': '📊 TABLE 5: AE + Radiomics',
        'fusion_type': 'features',
        'domains': ['AE', 'R']
    }
    
    configs['Table5_AE_C'] = {
        'description': '📊 TABLE 5: AE + Clinical',
        'fusion_type': 'features',
        'domains': ['AE', 'C']
    }
    
    configs['Table5_R_C'] = {
        'description': '📊 TABLE 5: Radiomics + Clinical',
        'fusion_type': 'features',
        'domains': ['R', 'C']
    }
    
    configs['Table5_AE_R_C'] = {
        'description': '📊 TABLE 5: AE + Radiomics + Clinical',
        'fusion_type': 'features',
        'domains': ['AE', 'R', 'C']
    }
    
    if has_dt:
        configs['Table5_DT_C'] = {
            'description': '📊 TABLE 5: DT + Clinical',
            'fusion_type': 'features',
            'domains': ['DT', 'C']
        }
        
        configs['Table5_DT_R'] = {
            'description': '📊 TABLE 5: DT + Radiomics',
            'fusion_type': 'features',
            'domains': ['DT', 'R']
        }
        
        configs['Table5_DT_R_C'] = {
            'description': '📊 TABLE 5: DT + Radiomics + Clinical',
            'fusion_type': 'features',
            'domains': ['DT', 'R', 'C']
        }
        
        configs['Table5_DT_AE_R_C'] = {
            'description': '📊 TABLE 5: DT + AE + Radiomics + Clinical (ALL)',
            'fusion_type': 'features',
            'domains': ['DT', 'AE', 'R', 'C']
        }
    
    # ========== TABLE 6: SIGNATURE-LEVEL FUSION (CoxPH only) ==========
    configs['Table6_AE_R'] = {
        'description': '⭐ TABLE 6: AE + Radiomics signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'R']
    }
    
    configs['Table6_AE_C'] = {
        'description': '⭐ TABLE 6: AE + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'C']
    }
    
    configs['Table6_R_C'] = {
        'description': '⭐ TABLE 6: Radiomics + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['R', 'C']
    }
    
    configs['Table6_AE_R_C'] = {
        'description': '⭐ TABLE 6: AE + Radiomics + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'R', 'C']
    }
    
    if has_dt:
        configs['Table6_DT_C'] = {
            'description': '⭐ TABLE 6: DT + Clinical signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'C']
        }
        
        configs['Table6_DT_R'] = {
            'description': '⭐ TABLE 6: DT + Radiomics signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'R']
        }
        
        configs['Table6_DT_R_C'] = {
            'description': '⭐ TABLE 6: DT + Radiomics + Clinical signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'R', 'C']
        }
        
        configs['Table6_DT_AE_R_C'] = {
            'description': '⭐ TABLE 6: DT + AE + R + C signatures (ALL)',
            'fusion_type': 'signatures',
            'domains': ['DT', 'AE', 'R', 'C']
        }
    
    # Domain feature mapping
    domain_features_map = {
        'C': clinical_features,
        'AE': ae_features,
        'R': radiomics_features,
        'DT': dt_features
    }
    
    domain_top_k_map = {
        'C': None,
        'AE': TOP_K_AE,
        'R': TOP_K_RADIOMICS,
        'DT': None
    }
    
    results = {}
    
    # Create KM plots directory
    if create_km_plots:
        km_dir = OUTPUT_DIR / "km_plots"
        km_dir.mkdir(parents=True, exist_ok=True)
    
    # Run each configuration with each model
    for config_name, config in configs.items():
        print(f"\n{'─' * 100}")
        print(f"🔬 Configuration: {config_name}")
        print(f"   {config['description']}")
        
        fusion_type = config.get('fusion_type', None)
        
        # For signature-level fusion, only use CoxPH (signatures are CoxPH-based)
        if fusion_type == 'signatures':
            models_for_config = ['coxph']
        else:
            models_for_config = models_to_run
        
        for model_type in models_for_config:
            result_key = f"{config_name}_{model_type}"
            
            print(f"\n   📈 Model: {model_type.upper()}")
            
            fold_metrics = {
                'c_index': [],
                'p_value': [],
                'hazard_ratio': [],
                'iauc': [],
                'iauc_min': [],
                'iauc_max': []
            }
            
            first_fold_data = None
            
            # Cross-validation
            for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_merged, df_merged['event'].astype(int)), 1):
                fold_train = df_merged.iloc[train_idx].copy()
                fold_val = df_merged.iloc[val_idx].copy()
                
                # ========== TABLE 5: FEATURE-LEVEL FUSION ==========
                if fusion_type == 'features':
                    domains = config['domains']
                    fold_features = []
                    
                    for domain in domains:
                        domain_feats = domain_features_map[domain]
                        top_k = domain_top_k_map[domain]
                        
                        if top_k is not None and len(domain_feats) > top_k:
                            selected = select_features_univariate_cox(fold_train, domain_feats, top_k)
                        else:
                            selected = domain_feats
                        
                        fold_features.extend(selected)
                    
                    # Remove duplicates
                    seen = set()
                    fold_features = [f for f in fold_features if not (f in seen or seen.add(f))]
                    
                    if len(fold_features) == 0:
                        continue
                    
                    # Train model and get predictions
                    risk_scores_val = train_and_predict(model_type, fold_train, fold_val, fold_features)
                    
                    if risk_scores_val is None:
                        continue
                
                # ========== TABLE 6: SIGNATURE-LEVEL FUSION ==========
                elif fusion_type == 'signatures':
                    domains = config['domains']
                    signatures_train = []
                    signatures_val = []
                    
                    for domain in domains:
                        domain_feats = domain_features_map[domain]
                        top_k = domain_top_k_map[domain]
                        
                        if top_k is not None and len(domain_feats) > top_k:
                            selected = select_features_univariate_cox(fold_train, domain_feats, top_k)
                        else:
                            selected = domain_feats
                        
                        if len(selected) == 0:
                            continue
                        
                        sig_train, sig_val = generate_domain_signature(fold_train, fold_val, selected, domain)
                        
                        if sig_train is not None:
                            signatures_train.append(sig_train)
                            signatures_val.append(sig_val)
                    
                    if len(signatures_train) == 0:
                        continue
                    
                    X_train_sigs = np.column_stack(signatures_train)
                    X_val_sigs = np.column_stack(signatures_val)
                    
                    sig_names = [f'{domains[i]}_Sig' for i in range(len(signatures_train))]
                    df_train_sigs = pd.DataFrame(X_train_sigs, columns=sig_names)
                    df_train_sigs['time_years'] = fold_train['time_years'].values
                    df_train_sigs['event'] = fold_train['event'].values
                    
                    df_val_sigs = pd.DataFrame(X_val_sigs, columns=sig_names)
                    
                    try:
                        cph_final = CoxPHFitter(penalizer=LASSO_PENALIZER)
                        cph_final.fit(df_train_sigs, duration_col='time_years', event_col='event')
                        
                        risk_scores_val = cph_final.predict_partial_hazard(df_val_sigs[sig_names]).values.flatten()
                    except:
                        continue
                
                # ========== SINGLE DOMAIN ==========
                else:
                    fold_features = config['features']
                    
                    if config.get('use_selection', False) and 'top_k' in config:
                        fold_features = select_features_univariate_cox(fold_train, fold_features, config['top_k'])
                    
                    if len(fold_features) == 0:
                        continue
                    
                    # Train model and get predictions
                    risk_scores_val = train_and_predict(model_type, fold_train, fold_val, fold_features)
                    
                    if risk_scores_val is None:
                        continue
                
                # Calculate metrics including iAUC
                metrics = calculate_all_metrics(
                    fold_val['time_years'].values,
                    fold_val['event'].values,
                    risk_scores_val,
                    time_train=fold_train['time_years'].values,
                    event_train=fold_train['event'].values
                )
                
                for key in fold_metrics:
                    fold_metrics[key].append(metrics.get(key, np.nan))
                
                # Store first fold for KM plots
                if fold_idx == 1 and create_km_plots:
                    first_fold_data = {
                        'time_val': fold_val['time_years'].values,
                        'event_val': fold_val['event'].values,
                        'risk_val': risk_scores_val
                    }
            
            # Calculate summary statistics
            if fold_metrics['c_index']:
                results[result_key] = {
                    'config_name': config_name,
                    'model_type': model_type,
                    'c_index_mean': np.nanmean(fold_metrics['c_index']),
                    'c_index_std': np.nanstd(fold_metrics['c_index']),
                    'p_value_mean': np.nanmean(fold_metrics['p_value']),
                    'p_value_std': np.nanstd(fold_metrics['p_value']),
                    'hr_mean': np.nanmean(fold_metrics['hazard_ratio']),
                    'hr_std': np.nanstd(fold_metrics['hazard_ratio']),
                    'iauc_mean': np.nanmean(fold_metrics['iauc']),
                    'iauc_std': np.nanstd(fold_metrics['iauc']),
                    'iauc_min_mean': np.nanmean(fold_metrics['iauc_min']),
                    'iauc_max_mean': np.nanmean(fold_metrics['iauc_max']),
                    'n_folds': len([x for x in fold_metrics['c_index'] if not np.isnan(x)]),
                    'description': config['description']
                }
                
                print(f"      ✅ C-Index: {results[result_key]['c_index_mean']:.4f} ± {results[result_key]['c_index_std']:.4f}")
                print(f"         p-value: {results[result_key]['p_value_mean']:.4f} ± {results[result_key]['p_value_std']:.4f}")
                print(f"         HR:      {results[result_key]['hr_mean']:.4f} ± {results[result_key]['hr_std']:.4f}")
                
                if not np.isnan(results[result_key]['iauc_mean']):
                    print(f"         iAUC:    {results[result_key]['iauc_mean']:.4f} ± {results[result_key]['iauc_std']:.4f} " +
                          f"[{results[result_key]['iauc_min_mean']:.3f}-{results[result_key]['iauc_max_mean']:.3f}]")
                else:
                    print(f"         iAUC:    N/A (requires scikit-survival)")
                
                # Create KM plot
                if create_km_plots and first_fold_data is not None:
                    km_path = km_dir / f"{result_key}_validation.png"
                    plot_km_curve(
                        first_fold_data['time_val'],
                        first_fold_data['event_val'],
                        first_fold_data['risk_val'],
                        f"{config_name} ({model_type})",
                        km_path,
                        "Validation"
                    )
    
    return results

# =============================================================================
# SENSITIVITY ANALYSIS FOR DIGITAL TWIN PARAMETERS
# =============================================================================

def perform_sensitivity_analysis(df_merged: pd.DataFrame,
                                 radiomics_features: List[str],
                                 ae_features: List[str],
                                 clinical_features: List[str],
                                 dt_features: List[str],
                                 output_dir: Path,
                                 perturbation_levels: List[float] = [0.1, 0.2],
                                 random_state: int = 42) -> Dict:
    """
    Perform sensitivity analysis by perturbing α and β parameters.
    
    Tests robustness of the Digital Twin model to parameter uncertainty.
    
    Parameters:
    -----------
    df_merged : DataFrame
        Complete merged dataset
    radiomics_features : list
        Radiomics feature names
    ae_features : list
        AutoEncoder feature names
    clinical_features : list
        Clinical feature names
    dt_features : list
        Digital Twin feature names
    output_dir : Path
        Directory to save results
    perturbation_levels : list
        List of perturbation fractions (e.g., [0.1, 0.2] for ±10%, ±20%)
    random_state : int
        Random seed
    
    Returns:
    --------
    results : dict
        Sensitivity analysis results
    """
    
    print("\n" + "="*100)
    print("🔬 SENSITIVITY ANALYSIS: Perturbation of Digital Twin Parameters (α, β)")
    print("="*100)
    
    if len(dt_features) == 0:
        print("   ⚠️  No Digital Twin features available. Skipping sensitivity analysis.")
        return {}
    
    # Create output directory
    sens_dir = output_dir / "sensitivity_analysis"
    sens_dir.mkdir(parents=True, exist_ok=True)
    
    # Use stratified split to get training and test sets
    from sklearn.model_selection import train_test_split
    
    df_train, df_test = train_test_split(
        df_merged, 
        test_size=0.2, 
        random_state=random_state,
        stratify=df_merged['event'].astype(int)
    )
    
    print(f"\n📊 Dataset Split:")
    print(f"   • Training set: {len(df_train)} patients")
    print(f"   • Test set: {len(df_test)} patients (for sensitivity analysis)")
    
    # Identify DT parameter columns
    if 'Bio_Alpha' not in df_train.columns or 'Bio_Necrosis' not in df_train.columns:
        print("   ⚠️  Bio_Alpha or Bio_Necrosis not found. Cannot perform sensitivity analysis.")
        return {}
    
    # ========== STEP 1: Train baseline model ==========
    print(f"\n{'─'*100}")
    print("STEP 1: Training Baseline Model (Full DT Configuration)")
    print(f"{'─'*100}")
    
    # Select features: top AE + top Radiomics + Clinical + all DT
    ae_selected = select_features_univariate_cox(df_train, ae_features, TOP_K_AE)
    rad_selected = select_features_univariate_cox(df_train, radiomics_features, TOP_K_RADIOMICS)
    
    baseline_features = clinical_features + ae_selected + rad_selected + dt_features
    
    # Remove duplicates
    seen = set()
    baseline_features = [f for f in baseline_features if not (f in seen or seen.add(f))]
    
    print(f"   • Total features: {len(baseline_features)}")
    print(f"     - Clinical: {len(clinical_features)}")
    print(f"     - AutoEncoder: {len(ae_selected)}")
    print(f"     - Radiomics: {len(rad_selected)}")
    print(f"     - Digital Twin: {len(dt_features)}")
    
    # Train baseline model
    scaler = StandardScaler()
    X_train = df_train[baseline_features].values
    X_test = df_test[baseline_features].values
    time_train = df_train['time_years'].values
    event_train = df_train['event'].values
    time_test = df_test['time_years'].values
    event_test = df_test['event'].values
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train CoxPH model
    df_train_scaled = df_train.copy()
    df_test_scaled = df_test.copy()
    df_train_scaled[baseline_features] = X_train_scaled
    df_test_scaled[baseline_features] = X_test_scaled
    
    baseline_cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
    baseline_cph.fit(
        df_train_scaled[baseline_features + ['time_years', 'event']],
        duration_col='time_years',
        event_col='event'
    )
    
    # Get baseline predictions
    baseline_risk = baseline_cph.predict_partial_hazard(
        df_test_scaled[baseline_features]
    ).values.flatten()
    
    baseline_metrics = calculate_all_metrics(
        time_test, event_test, baseline_risk,
        time_train, event_train
    )
    
    print(f"\n   ✅ Baseline Model Performance:")
    print(f"      • C-Index: {baseline_metrics['c_index']:.4f}")
    print(f"      • p-value: {baseline_metrics['p_value']:.6f}")
    print(f"      • HR: {baseline_metrics['hazard_ratio']:.4f}")
    if not np.isnan(baseline_metrics['iauc']):
        print(f"      • iAUC: {baseline_metrics['iauc']:.4f}")
    
    # ========== STEP 2: Perform perturbations ==========
    print(f"\n{'─'*100}")
    print("STEP 2: Perturbing Parameters and Computing New Risk Scores")
    print(f"{'─'*100}")
    
    results = {
        'baseline_risk': baseline_risk,
        'baseline_metrics': baseline_metrics,
        'perturbations': {}
    }
    
    # Define perturbation scenarios
    scenarios = []
    for level in perturbation_levels:
        scenarios.extend([
            (f'alpha_plus_{int(level*100)}', 'Bio_Alpha', +level),
            (f'alpha_minus_{int(level*100)}', 'Bio_Alpha', -level),
            (f'beta_plus_{int(level*100)}', 'Bio_Necrosis', +level),
            (f'beta_minus_{int(level*100)}', 'Bio_Necrosis', -level),
            (f'both_plus_{int(level*100)}', 'both', +level),
            (f'both_minus_{int(level*100)}', 'both', -level),
        ])
    
    print(f"\n   📋 Perturbation Scenarios: {len(scenarios)}")
    
    for scenario_name, param_name, delta_fraction in tqdm(scenarios, desc="   Running scenarios"):
        
        # Create perturbed test set
        df_test_perturbed = df_test.copy()
        
        if param_name == 'Bio_Alpha':
            df_test_perturbed['Bio_Alpha'] = df_test['Bio_Alpha'] * (1 + delta_fraction)
            # Assume growth rate scales linearly with alpha
            if 'Sim_Growth_Rate' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Growth_Rate'] = df_test['Sim_Growth_Rate'] * (1 + delta_fraction)
        
        elif param_name == 'Bio_Necrosis':
            df_test_perturbed['Bio_Necrosis'] = df_test['Bio_Necrosis'] * (1 + delta_fraction)
            # Assume necrosis ratio scales with beta
            if 'Sim_Necrosis_Ratio' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test['Sim_Necrosis_Ratio'] * (1 + delta_fraction)
                # Clip to valid range [0, 1]
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test_perturbed['Sim_Necrosis_Ratio'].clip(0, 1)
        
        elif param_name == 'both':
            df_test_perturbed['Bio_Alpha'] = df_test['Bio_Alpha'] * (1 + delta_fraction)
            df_test_perturbed['Bio_Necrosis'] = df_test['Bio_Necrosis'] * (1 + delta_fraction)
            if 'Sim_Growth_Rate' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Growth_Rate'] = df_test['Sim_Growth_Rate'] * (1 + delta_fraction)
            if 'Sim_Necrosis_Ratio' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test['Sim_Necrosis_Ratio'] * (1 + delta_fraction)
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test_perturbed['Sim_Necrosis_Ratio'].clip(0, 1)
        
        # Scale perturbed features
        X_test_perturbed = df_test_perturbed[baseline_features].values
        X_test_perturbed_scaled = scaler.transform(X_test_perturbed)
        df_test_perturbed_scaled = df_test_perturbed.copy()
        df_test_perturbed_scaled[baseline_features] = X_test_perturbed_scaled
        
        # Predict with perturbed features
        perturbed_risk = baseline_cph.predict_partial_hazard(
            df_test_perturbed_scaled[baseline_features]
        ).values.flatten()
        
        # Compute metrics for perturbed predictions
        perturbed_metrics = calculate_all_metrics(
            time_test, event_test, perturbed_risk,
            time_train, event_train
        )
        
        # Analyze differences
        risk_diff = perturbed_risk - baseline_risk
        risk_diff_pct = (risk_diff / (baseline_risk + 1e-8)) * 100
        
        # Rank correlation
        from scipy.stats import spearmanr, kendalltau
        spearman_rho, spearman_p = spearmanr(baseline_risk, perturbed_risk)
        kendall_tau, kendall_p = kendalltau(baseline_risk, perturbed_risk)
        
        # Risk category changes (stratified by median)
        baseline_median = np.median(baseline_risk)
        perturbed_median = np.median(perturbed_risk)
        
        baseline_high_risk = baseline_risk >= baseline_median
        perturbed_high_risk = perturbed_risk >= perturbed_median
        
        risk_category_changes = (baseline_high_risk != perturbed_high_risk).sum()
        pct_category_changes = 100 * risk_category_changes / len(baseline_risk)
        
        # Store results
        results['perturbations'][scenario_name] = {
            'param_name': param_name,
            'delta_fraction': delta_fraction,
            'delta_pct': delta_fraction * 100,
            'perturbed_risk': perturbed_risk,
            'risk_diff_mean': np.mean(risk_diff),
            'risk_diff_std': np.std(risk_diff),
            'risk_diff_abs_mean': np.mean(np.abs(risk_diff)),
            'risk_diff_pct_mean': np.mean(risk_diff_pct),
            'risk_diff_pct_abs_mean': np.mean(np.abs(risk_diff_pct)),
            'spearman_rho': spearman_rho,
            'spearman_p': spearman_p,
            'kendall_tau': kendall_tau,
            'kendall_p': kendall_p,
            'risk_category_changes': risk_category_changes,
            'pct_category_changes': pct_category_changes,
            'c_index': perturbed_metrics['c_index'],
            'c_index_diff': perturbed_metrics['c_index'] - baseline_metrics['c_index'],
            'p_value': perturbed_metrics['p_value'],
            'hazard_ratio': perturbed_metrics['hazard_ratio'],
            'iauc': perturbed_metrics['iauc']
        }
    
    # ========== STEP 3: Summarize results ==========
    print(f"\n{'─'*100}")
    print("STEP 3: Sensitivity Analysis Summary")
    print(f"{'─'*100}")
    
    # Create summary table
    summary_rows = []
    for scenario_name, scenario_data in results['perturbations'].items():
        summary_rows.append({
            'Scenario': scenario_name,
            'Parameter': scenario_data['param_name'],
            'Δ (%)': f"{scenario_data['delta_pct']:+.0f}%",
            'ΔRisk (abs mean)': scenario_data['risk_diff_abs_mean'],
            'ΔRisk (% mean)': scenario_data['risk_diff_pct_abs_mean'],
            'Spearman ρ': scenario_data['spearman_rho'],
            'Category Changes (%)': scenario_data['pct_category_changes'],
            'ΔC-Index': scenario_data['c_index_diff'],
            'C-Index': scenario_data['c_index']
        })
    
    df_summary = pd.DataFrame(summary_rows)
    
    # Save summary table
    summary_path = sens_dir / "sensitivity_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\n   ✅ Summary saved to: {summary_path}")
    
    # Print summary table
    print(f"\n   📊 SENSITIVITY ANALYSIS RESULTS:")
    print(f"   {'─'*98}")
    print(f"   {'Scenario':<22} {'Δ':<8} {'ΔRisk(abs)':<12} {'Spearman ρ':<12} {'Cat.Chg(%)':<12} {'ΔC-Idx':<10}")
    print(f"   {'─'*98}")
    
    for _, row in df_summary.iterrows():
        print(f"   {row['Scenario']:<22} {row['Δ (%)']:<8} "
              f"{row['ΔRisk (abs mean)']:>11.4f} {row['Spearman ρ']:>11.4f} "
              f"{row['Category Changes (%)']:>11.1f} {row['ΔC-Index']:>9.4f}")
    
    print(f"   {'─'*98}")
    
    # ========== STEP 4: Create visualizations ==========
    print(f"\n{'─'*100}")
    print("STEP 4: Creating Visualizations")
    print(f"{'─'*100}")
    
    # Plot 1: Risk score changes for each perturbation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Sensitivity Analysis: Risk Score Changes', fontsize=16, fontweight='bold')
    
    # Alpha perturbations
    ax = axes[0, 0]
    for level in perturbation_levels:
        for sign, marker in [('+', 'o'), ('-', 's')]:
            key = f'alpha_{sign.replace("+", "plus").replace("-", "minus")}_{int(level*100)}'
            if key in results['perturbations']:
                data = results['perturbations'][key]
                risk_diffs = data['perturbed_risk'] - baseline_risk
                ax.scatter(baseline_risk, risk_diffs, alpha=0.5, s=30, marker=marker,
                          label=f'Δα={sign}{int(level*100)}%')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Baseline Risk Score', fontsize=11, fontweight='bold')
    ax.set_ylabel('Change in Risk Score', fontsize=11, fontweight='bold')
    ax.set_title('α (Proliferation) Perturbations', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='best')
    ax.grid(alpha=0.3)
    
    # Beta perturbations
    ax = axes[0, 1]
    for level in perturbation_levels:
        for sign, marker in [('+', 'o'), ('-', 's')]:
            key = f'beta_{sign.replace("+", "plus").replace("-", "minus")}_{int(level*100)}'
            if key in results['perturbations']:
                data = results['perturbations'][key]
                risk_diffs = data['perturbed_risk'] - baseline_risk
                ax.scatter(baseline_risk, risk_diffs, alpha=0.5, s=30, marker=marker,
                          label=f'Δβ={sign}{int(level*100)}%')
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('Baseline Risk Score', fontsize=11, fontweight='bold')
    ax.set_ylabel('Change in Risk Score', fontsize=11, fontweight='bold')
    ax.set_title('β (Necrosis) Perturbations', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='best')
    ax.grid(alpha=0.3)
    
    # Rank correlation plot
    ax = axes[1, 0]
    scenarios_sorted = sorted(results['perturbations'].items(), 
                             key=lambda x: abs(x[1]['delta_fraction']))
    scenario_names = [s[0] for s in scenarios_sorted]
    spearman_rhos = [s[1]['spearman_rho'] for s in scenarios_sorted]
    
    colors = ['red' if 'alpha' in name else 'blue' if 'beta' in name else 'green' 
              for name in scenario_names]
    
    bars = ax.barh(range(len(scenario_names)), spearman_rhos, color=colors, alpha=0.7)
    ax.set_yticks(range(len(scenario_names)))
    ax.set_yticklabels(scenario_names, fontsize=8)
    ax.set_xlabel('Spearman Rank Correlation (ρ)', fontsize=11, fontweight='bold')
    ax.set_title('Rank Preservation After Perturbation', fontsize=12, fontweight='bold')
    ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1)
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim([0.9, 1.0])
    
    # Category changes plot
    ax = axes[1, 1]
    pct_changes = [s[1]['pct_category_changes'] for s in scenarios_sorted]
    
    bars = ax.barh(range(len(scenario_names)), pct_changes, color=colors, alpha=0.7)
    ax.set_yticks(range(len(scenario_names)))
    ax.set_yticklabels(scenario_names, fontsize=8)
    ax.set_xlabel('Risk Category Changes (%)', fontsize=11, fontweight='bold')
    ax.set_title('Patients Changing Risk Category', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plot_path = sens_dir / "sensitivity_analysis_plots.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Plots saved to: {plot_path}")
    
    # Plot 2: C-Index changes
    fig, ax = plt.subplots(figsize=(10, 6))
    c_index_diffs = [s[1]['c_index_diff'] for s in scenarios_sorted]
    
    bars = ax.barh(range(len(scenario_names)), c_index_diffs, color=colors, alpha=0.7)
    ax.set_yticks(range(len(scenario_names)))
    ax.set_yticklabels(scenario_names, fontsize=9)
    ax.set_xlabel('ΔC-Index', fontsize=12, fontweight='bold')
    ax.set_title('Impact on Model Discrimination (C-Index Change)', 
                 fontsize=13, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    cindex_plot_path = sens_dir / "c_index_sensitivity.png"
    plt.savefig(cindex_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   ✅ C-Index plot saved to: {cindex_plot_path}")
    
    # ========== STEP 5: Key findings ==========
    print(f"\n{'─'*100}")
    print("STEP 5: Key Findings")
    print(f"{'─'*100}")
    
    # Find most sensitive parameter
    alpha_sensitivity = np.mean([
        results['perturbations'][k]['risk_diff_abs_mean']
        for k in results['perturbations'].keys() if 'alpha' in k
    ])
    
    beta_sensitivity = np.mean([
        results['perturbations'][k]['risk_diff_abs_mean']
        for k in results['perturbations'].keys() if 'beta' in k
    ])
    
    print(f"\n   📈 Average Risk Score Change (absolute):")
    print(f"      • α perturbations: {alpha_sensitivity:.4f}")
    print(f"      • β perturbations: {beta_sensitivity:.4f}")
    
    if alpha_sensitivity > beta_sensitivity:
        print(f"      → Model is MORE sensitive to α (proliferation rate)")
    else:
        print(f"      → Model is MORE sensitive to β (necrosis probability)")
    
    # Rank preservation
    avg_spearman = np.mean([d['spearman_rho'] for d in results['perturbations'].values()])
    min_spearman = np.min([d['spearman_rho'] for d in results['perturbations'].values()])
    
    print(f"\n   📊 Rank Preservation:")
    print(f"      • Average Spearman ρ: {avg_spearman:.4f}")
    print(f"      • Minimum Spearman ρ: {min_spearman:.4f}")
    print(f"      → Rankings are {'HIGHLY' if min_spearman > 0.95 else 'MODERATELY'} stable")
    
    # Category changes
    avg_category_changes = np.mean([d['pct_category_changes'] for d in results['perturbations'].values()])
    max_category_changes = np.max([d['pct_category_changes'] for d in results['perturbations'].values()])
    
    print(f"\n   🔄 Risk Category Changes:")
    print(f"      • Average: {avg_category_changes:.1f}% of patients")
    print(f"      • Maximum: {max_category_changes:.1f}% of patients")
    
    # C-Index stability
    avg_c_index_change = np.mean([abs(d['c_index_diff']) for d in results['perturbations'].values()])
    max_c_index_change = np.max([abs(d['c_index_diff']) for d in results['perturbations'].values()])
    
    print(f"\n   📉 C-Index Stability:")
    print(f"      • Average |ΔC-Index|: {avg_c_index_change:.4f}")
    print(f"      • Maximum |ΔC-Index|: {max_c_index_change:.4f}")
    
    if max_c_index_change < 0.02:
        print(f"      → Model performance is HIGHLY ROBUST to parameter perturbations")
    elif max_c_index_change < 0.05:
        print(f"      → Model performance is MODERATELY ROBUST")
    else:
        print(f"      → Model shows SENSITIVITY to parameter perturbations")
    
    print(f"\n{'='*100}")
    print("✅ SENSITIVITY ANALYSIS COMPLETE")
    print(f"{'='*100}")
    
    # Save full results
    results_path = sens_dir / "sensitivity_analysis_full_results.pkl"
    import pickle
    with open(results_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"\n   💾 Full results saved to: {results_path}")
    
    return results




# =============================================================================
# RESULTS DISPLAY AND SAVING
# =============================================================================

def print_results_summary(results: Dict):
    """Print comprehensive results summary including iAUC and multiple models."""
    
    print("\n" + "="*100)
    print("📊 FINAL RESULTS SUMMARY WITH INTEGRATED AUC AND MULTIPLE MODELS")
    print("="*100)
    
    # Sort by C-Index
    sorted_results = sorted(results.items(), key=lambda x: x[1]['c_index_mean'], reverse=True)
    
    # Check if we have iAUC data
    has_iauc = any(not np.isnan(r[1]['iauc_mean']) for r in sorted_results)
    
    # Print results table
    header = "   ┌─────────────────────────────────────────┬──────────┬────────────────┬────────────┬────────────"
    if has_iauc:
        header += "┬────────────────┐"
    else:
        header += "┐"
    print(f"\n{header}")
    
    col_header = "   │ Configuration                           │ Model    │ C-Index        │ p-value    │ HR         "
    if has_iauc:
        col_header += "│ iAUC           │"
    else:
        col_header += "│"
    print(col_header)
    
    sep = "   ├─────────────────────────────────────────┼──────────┼────────────────┼────────────┼────────────"
    if has_iauc:
        sep += "┼────────────────┤"
    else:
        sep += "┤"
    print(sep)
    
    for result_key, data in sorted_results:
        config_name = data.get('config_name', result_key.rsplit('_', 1)[0])
        model_type = data.get('model_type', 'coxph')
        
        name_padded = config_name[:39].ljust(39)
        model_padded = model_type[:8].ljust(8)
        c_str = f"{data['c_index_mean']:.4f}±{data['c_index_std']:.4f}".center(14)
        p_str = f"{data['p_value_mean']:.4f}".center(10)
        hr_str = f"{data['hr_mean']:.2f}±{data['hr_std']:.2f}".center(10)
        
        if has_iauc:
            if not np.isnan(data['iauc_mean']):
                iauc_str = f"{data['iauc_mean']:.4f}±{data['iauc_std']:.4f}".center(14)
            else:
                iauc_str = "N/A".center(14)
            print(f"   │ {name_padded} │ {model_padded} │ {c_str} │ {p_str} │ {hr_str} │ {iauc_str} │")
        else:
            print(f"   │ {name_padded} │ {model_padded} │ {c_str} │ {p_str} │ {hr_str} │")
    
    footer = "   └─────────────────────────────────────────┴──────────┴────────────────┴────────────┴────────────"
    if has_iauc:
        footer += "┴────────────────┘"
    else:
        footer += "┘"
    print(footer)
    
    # Model comparison
    print("\n   📊 MODEL COMPARISON (averaged across configurations):")
    model_summary = {}
    for result_key, data in results.items():
        model_type = data.get('model_type', 'coxph')
        if model_type not in model_summary:
            model_summary[model_type] = {'c_indices': [], 'iaucs': []}
        model_summary[model_type]['c_indices'].append(data['c_index_mean'])
        if not np.isnan(data['iauc_mean']):
            model_summary[model_type]['iaucs'].append(data['iauc_mean'])
    
    for model_type, summary in sorted(model_summary.items(), 
                                       key=lambda x: np.mean(x[1]['c_indices']), 
                                       reverse=True):
        c_avg = np.mean(summary['c_indices'])
        c_std = np.std(summary['c_indices'])
        iauc_avg = np.mean(summary['iaucs']) if summary['iaucs'] else np.nan
        
        if not np.isnan(iauc_avg):
            print(f"      • {model_type.upper()}: Avg C-Index={c_avg:.4f}±{c_std:.4f}, Avg iAUC={iauc_avg:.4f}")
        else:
            print(f"      • {model_type.upper()}: Avg C-Index={c_avg:.4f}±{c_std:.4f}")
    
    # Highlight Digital Twin results
    dt_configs = [key for key in results.keys() if 'DT' in key]
    if dt_configs:
        print("\n   🔬 DIGITAL TWIN (3-D MLPA) RESULTS:")
        for result_key in sorted(dt_configs, key=lambda x: results[x]['c_index_mean'], reverse=True)[:5]:
            data = results[result_key]
            model = data.get('model_type', 'coxph')
            config = data.get('config_name', result_key)
            if has_iauc and not np.isnan(data['iauc_mean']):
                print(f"      • {config} ({model}): C={data['c_index_mean']:.4f}±{data['c_index_std']:.4f}, " +
                      f"HR={data['hr_mean']:.2f}, iAUC={data['iauc_mean']:.4f}")
            else:
                print(f"      • {config} ({model}): C={data['c_index_mean']:.4f}±{data['c_index_std']:.4f}, " +
                      f"HR={data['hr_mean']:.2f}")
    
    # Best configuration
    best_key = sorted_results[0][0]
    best_data = sorted_results[0][1]
    best_c = best_data['c_index_mean']
    best_model = best_data.get('model_type', 'coxph')
    best_config = best_data.get('config_name', best_key)
    
    print(f"\n   🏆 Best by C-Index: {best_config} ({best_model}) - C-Index: {best_c:.4f}")
    
    if has_iauc:
        sorted_by_iauc = sorted([(k, v) for k, v in results.items() if not np.isnan(v['iauc_mean'])], 
                                key=lambda x: x[1]['iauc_mean'], reverse=True)
        if sorted_by_iauc:
            best_iauc_key = sorted_by_iauc[0][0]
            best_iauc_data = sorted_by_iauc[0][1]
            best_iauc = best_iauc_data['iauc_mean']
            best_iauc_model = best_iauc_data.get('model_type', 'coxph')
            best_iauc_config = best_iauc_data.get('config_name', best_iauc_key)
            print(f"   🏆 Best by iAUC: {best_iauc_config} ({best_iauc_model}) - iAUC: {best_iauc:.4f}")


def save_results_to_csv(results: Dict, output_dir: Path):
    """Save results to CSV file including iAUC and model type."""
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    rows = []
    for result_key, data in results.items():
        config_name = data.get('config_name', result_key.rsplit('_', 1)[0])
        model_type = data.get('model_type', 'coxph')
        
        # Determine fusion type
        if 'Table5' in config_name:
            fusion_type = 'Table 5 (Feature Fusion)'
        elif 'Table6' in config_name:
            fusion_type = 'Table 6 (Signature Fusion)'
        else:
            fusion_type = 'Single Domain'
        
        # Check if DT is included
        has_dt = 'DT' in config_name
        
        rows.append({
            'Result_Key': result_key,
            'Configuration': config_name,
            'Model_Type': model_type,
            'Fusion_Type': fusion_type,
            'Includes_DT': has_dt,
            'C_Index_Mean': data['c_index_mean'],
            'C_Index_Std': data['c_index_std'],
            'P_Value_Mean': data['p_value_mean'],
            'P_Value_Std': data['p_value_std'],
            'HR_Mean': data['hr_mean'],
            'HR_Std': data['hr_std'],
            'iAUC_Mean': data['iauc_mean'],
            'iAUC_Std': data['iauc_std'],
            'iAUC_Min_Mean': data['iauc_min_mean'],
            'iAUC_Max_Mean': data['iauc_max_mean'],
            'N_Folds': data['n_folds'],
            'Description': data['description']
        })
    
    df = pd.DataFrame(rows).sort_values('C_Index_Mean', ascending=False)
    csv_path = output_dir / "results_with_3d_mlpa_iauc_multimodel.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n✅ Results saved to: {csv_path}")
    
    # Create pivot table by model
    pivot_path = output_dir / "results_pivot_by_model.csv"
    df_pivot = df.pivot_table(
        index='Configuration', 
        columns='Model_Type', 
        values=['C_Index_Mean', 'iAUC_Mean'],
        aggfunc='first'
    ).round(4)
    df_pivot.to_csv(pivot_path)
    print(f"✅ Pivot table saved to: {pivot_path}")
    
    # Also create a summary by iAUC
    if not df['iAUC_Mean'].isna().all():
        df_iauc = df.sort_values('iAUC_Mean', ascending=False)
        csv_iauc_path = output_dir / "results_sorted_by_iauc.csv"
        df_iauc.to_csv(csv_iauc_path, index=False)
        print(f"✅ iAUC-sorted results saved to: {csv_iauc_path}")
    
    return df


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main execution function."""
    
    start_time = time.time()
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "="*100)
    print("🚀 FERRETTI REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS")
    print("="*100)
    print(f"\n📁 Output Directory: {OUTPUT_DIR}/")
    print(f"📊 Cross-Validation: {N_SPLITS}-fold")
    
    print("\n🔧 AVAILABLE MODELS:")
    print(f"   1. CoxPH (lifelines): ✅ Always available")
    if TORCH_AVAILABLE:
        print(f"   2. Neural Cox (PyTorch): ✅ Available")
        print(f"      • Hidden layers: {NEURAL_COX_HIDDEN_LAYERS}")
        print(f"      • Dropout: {NEURAL_COX_DROPOUT}")
        print(f"      • Epochs: {NEURAL_COX_EPOCHS}")
    else:
        print(f"   2. Neural Cox (PyTorch): ❌ Not available (pip install torch)")
    
    if SKSURV_AVAILABLE:
        print(f"   3. Gradient Boosting (scikit-survival): ✅ Available")
        print(f"   4. iAUC calculation: ✅ Available")
        print(f"      • Time points: {IAUC_TIME_POINTS.tolist()}")
    else:
        print(f"   3. Gradient Boosting: ❌ Not available (pip install scikit-survival)")
        print(f"   4. iAUC calculation: ❌ Not available")
    
    print(f"   5. Ensemble: {'✅ Available' if (TORCH_AVAILABLE or SKSURV_AVAILABLE) else '⚠️ CoxPH only'}")
    if TORCH_AVAILABLE or SKSURV_AVAILABLE:
        print(f"      • Weights: {ENSEMBLE_WEIGHTS}")
    
    print("\n" + "─"*100)
    print("🧬 3-D MLPA DIGITAL TWIN EXPLANATION:")
    print("─"*100)
    print("""
    The Digital Twin uses Multi-Level Parameterized Automata (MLPA) to simulate
    tumor growth based on radiomics-derived biological parameters:
    
    1. RADIOMICS → BIOLOGICAL PARAMETERS:
       • Entropy → α (proliferation): Higher entropy = more heterogeneous = faster growth
         Formula: α = 0.01 + 0.05 * normalized_entropy
       
       • Sphericity → β (necrosis): Lower sphericity = irregular shape = more necrosis
         Formula: β = 0.01 + 0.08 * (1 - sphericity)
    
    2. CELLULAR AUTOMATON SIMULATION:
       • Grid states: Lung(1), Tumor(2), Necrotic(3), Proliferating(4), Malignant(6)
       • Growth: Tumor cells proliferate at rate α into adjacent lung tissue
       • Necrosis: Core cells die at rate β when oxygen-deprived
       • Mutation: Some cells become malignant (more aggressive)
    
    3. OUTPUT FEATURES:
       • Sim_Growth_Rate: Overall tumor expansion rate
       • Sim_Necrosis_Ratio: Fraction of dead cells in tumor
       • Bio_Alpha, Bio_Necrosis: Biological parameters
       • Radiomics_Entropy, Radiomics_Sphericity: Base radiomics
    
    4. MODEL TYPES:
       • CoxPH: Classical linear proportional hazards model
       • Neural Cox: Deep learning-based survival prediction (DeepSurv-style)
       • Ensemble: Weighted combination of multiple models
    
    5. INTEGRATED AUC (iAUC):
       • Measures time-dependent discrimination ability
       • Evaluates AUC at multiple time points (0.25, 0.5, ..., 2.0 years)
       • Takes mean of time-specific AUCs
       • More comprehensive than single-time-point C-index
    """)
    print("─"*100)
    
    # ========== Load All Data ==========
    df_clinical = load_clinical_data(CLINICAL_FILE)
    
    clinical_features = [
        'age', 'gender_encoded', 'T_Stage_encoded', 'N_Stage_encoded',
        'M_Stage_encoded', 'Overall_Stage_encoded', 'Histology_encoded'
    ]
    
    df_radiomics, radiomics_features, patient_ids_rad = load_radiomics_from_npy(
        RADIOMICS_NPY_FILE, RADIOMICS_NAMES_JSON, PATIENT_IDS_JSON
    )
    
    df_ae, ae_features = load_ae_features(DEEP_FEATURES_FILE)
    
    df_dt, dt_features = load_digital_twin_features(DT_FEATURES_FILE)
    
    # ========== Merge All Data ==========
    print("\n" + "="*100)
    print("📊 MERGING ALL DATA")
    print("="*100)
    
    df_clin_indexed = df_clinical.set_index('PatientID')
    df_rad_indexed = df_radiomics.set_index('PatientID')
    df_ae_indexed = df_ae.set_index('PatientID')
    
    df_merged = df_clin_indexed.join(df_rad_indexed, how='inner')
    df_merged = df_merged.join(df_ae_indexed, how='inner')
    
    if len(dt_features) > 0 and len(df_dt) > 0:
        df_dt_indexed = df_dt.set_index('PatientID')
        df_merged = df_merged.join(df_dt_indexed, how='inner')
        print(f"   ✅ Added Digital Twin features (inner join)")
    
    df_merged = df_merged.dropna(subset=['time_years', 'event'])
    
    print(f"\n✅ Final merged dataset: {len(df_merged)} patients")
    print(f"\n📊 Feature Summary:")
    print(f"   • Clinical:      {len(clinical_features)} features")
    print(f"   • Radiomics:     {len(radiomics_features)} features")
    print(f"   • AutoEncoder:   {len(ae_features)} features")
    print(f"   • Digital Twin:  {len(dt_features)} features")
    
    # Apply feature selection pipeline
    print("\n📊 Applying Ferretti Feature Selection Pipeline:")
    radiomics_selected = remove_correlated_features(df_merged, radiomics_features, CORRELATION_THRESHOLD)
    radiomics_selected = remove_constant_features(df_merged, radiomics_selected)
    print(f"   • Radiomics: {len(radiomics_features)} → {len(radiomics_selected)} features")
    
    if len(dt_features) > 0:
        dt_selected = remove_correlated_features(df_merged, dt_features, CORRELATION_THRESHOLD)
        dt_selected = remove_constant_features(df_merged, dt_selected)
        print(f"   • Digital Twin: {len(dt_features)} → {len(dt_selected)} features")
    else:
        dt_selected = []
    
    # ========== Run Experiment ==========
    results = run_experiment_with_3d_mlpa(
        df_merged=df_merged,
        radiomics_features=radiomics_selected,
        ae_features=ae_features,
        clinical_features=clinical_features,
        dt_features=dt_selected,
        n_splits=N_SPLITS,
        random_state=RANDOM_STATE,
        create_km_plots=CREATE_KM_PLOTS
    )
    # =============================================================================
    # Update the main() function to include sensitivity analysis
    # =============================================================================

    # Add this at the end of main(), right before the final summary:

    # ========== Run Sensitivity Analysis ==========
    if len(dt_selected) > 0:
        print("\n" + "="*100)
        print("🔬 RUNNING SENSITIVITY ANALYSIS")
        print("="*100)
        
        sensitivity_results = perform_sensitivity_analysis(
            df_merged=df_merged,
            radiomics_features=radiomics_selected,
            ae_features=ae_features,
            clinical_features=clinical_features,
            dt_features=dt_selected,
            output_dir=OUTPUT_DIR,
            perturbation_levels=[0.1, 0.2],  # ±10%, ±20%
            random_state=RANDOM_STATE
        )
    else:
        print("\n⚠️  Skipping sensitivity analysis (no Digital Twin features available)")    
    
    # ========== Display & Save Results ==========
    print_results_summary(results)
    save_results_to_csv(results, OUTPUT_DIR)
    
    # Final summary
    end_time = time.time()
    duration = end_time - start_time
    
    print("\n" + "="*100)
    print("✅ ANALYSIS COMPLETE WITH NEURAL COX AND ENSEMBLE!")
    print("="*100)
    print(f"\n⏱️  Total Runtime: {duration/60:.1f} minutes")
    print(f"\n📁 Output Files:")
    print(f"   • Results CSV: {OUTPUT_DIR}/results_with_3d_mlpa_iauc_multimodel.csv")
    print(f"   • Pivot table: {OUTPUT_DIR}/results_pivot_by_model.csv")
    if SKSURV_AVAILABLE:
        print(f"   • iAUC-sorted: {OUTPUT_DIR}/results_sorted_by_iauc.csv")
    if CREATE_KM_PLOTS:
        print(f"   • KM Plots:    {OUTPUT_DIR}/km_plots/")
    
    # Count results by model type
    model_counts = {}
    for result_key, data in results.items():
        model = data.get('model_type', 'coxph')
        model_counts[model] = model_counts.get(model, 0) + 1
    
    print("\n📊 Results by Model Type:")
    for model, count in sorted(model_counts.items()):
        print(f"   • {model.upper()}: {count} configurations")
    print(f"   • Total: {len(results)}")
    
    print("\n🔬 Key Findings:")
    
    # Best overall
    best = max(results.items(), key=lambda x: x[1]['c_index_mean'])
    print(f"   • Best overall: {best[1].get('config_name', best[0])} ({best[1].get('model_type', 'coxph')})")
    print(f"     C-Index = {best[1]['c_index_mean']:.4f}", end="")
    if not np.isnan(best[1]['iauc_mean']):
        print(f", iAUC = {best[1]['iauc_mean']:.4f}")
    else:
        print()
    
    # Best by model
    print("\n   • Best by model type:")
    for model in ['coxph', 'neural_cox', 'ensemble']:
        model_results = [(k, v) for k, v in results.items() if v.get('model_type', 'coxph') == model]
        if model_results:
            best_model = max(model_results, key=lambda x: x[1]['c_index_mean'])
            print(f"     - {model.upper()}: {best_model[1].get('config_name', best_model[0])} " +
                  f"(C={best_model[1]['c_index_mean']:.4f})")
    
    if SKSURV_AVAILABLE:
        print("\n📈 iAUC Interpretation:")
        print("   • iAUC > 0.70: Excellent time-dependent discrimination")
        print("   • iAUC 0.60-0.70: Good discrimination")
        print("   • iAUC 0.50-0.60: Moderate discrimination")
        print("   • iAUC = 0.50: Random prediction")
    
    if not TORCH_AVAILABLE:
        print("\n⚠️  To enable Neural Cox:")
        print("   pip install torch")
    
    if not SKSURV_AVAILABLE:
        print("\n⚠️  To enable iAUC and Gradient Boosting:")
        print("   pip install scikit-survival")
    
    print("\n" + "="*100)


if __name__ == "__main__":
    main()


🚀 FERRETTI REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS

📁 Output Directory: ferretti_with_3d_mlpa_iauc_output/
📊 Cross-Validation: 5-fold

🔧 AVAILABLE MODELS:
   1. CoxPH (lifelines): ✅ Always available
   2. Neural Cox (PyTorch): ✅ Available
      • Hidden layers: [64, 32]
      • Dropout: 0.3
      • Epochs: 100
   3. Gradient Boosting (scikit-survival): ✅ Available
   4. iAUC calculation: ✅ Available
      • Time points: [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
   5. Ensemble: ✅ Available
      • Weights: {'coxph': 0.4, 'neural_cox': 0.3, 'gradient_boosting': 0.3}

────────────────────────────────────────────────────────────────────────────────────────────────────
🧬 3-D MLPA DIGITAL TWIN EXPLANATION:
────────────────────────────────────────────────────────────────────────────────────────────────────

    The Digital Twin uses Multi-Level Parameterized Automata (MLPA) to simulate
    tumor growth based on radiomics-derived biological parameters:
    
    1. R


────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: DT_Only
   Digital Twin 3-D MLPA (4 features)

   📈 Model: COXPH
      ✅ C-Index: 0.5737 ± 0.0324
         p-value: 0.3376 ± 0.2760
         HR:      1.2197 ± 0.2563
         iAUC:    0.6207 ± 0.0450 [0.576-0.690]

   📈 Model: NEURAL_COX
      ✅ C-Index: 0.5600 ± 0.0370
         p-value: 0.1660 ± 0.1072
         HR:      1.2569 ± 0.3192
         iAUC:    0.5969 ± 0.0565 [0.551-0.648]

   📈 Model: ENSEMBLE
      ✅ C-Index: 0.5469 ± 0.0360
         p-value: 0.4015 ± 0.3781
         HR:      1.1592 ± 0.3270
         iAUC:    0.5781 ± 0.0429 [0.531-0.648]

────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: Table5_AE_R
   📊 TABLE 5: AE + Radiomics

   📈 Model: COXPH
      ✅ C-Index: 0.5884 ± 0.0327
         p-value: 0.2255 ± 0.3231
         HR:      1.5141 ± 0.3278
         iAUC:    0.6458 ± 0.0520 [0.595-0.


────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: Table6_DT_R_C
   ⭐ TABLE 6: DT + Radiomics + Clinical signatures

   📈 Model: COXPH
      ✅ C-Index: 0.6115 ± 0.0258
         p-value: 0.0188 ± 0.0211
         HR:      1.9444 ± 0.3464
         iAUC:    0.6671 ± 0.0337 [0.609-0.738]

────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: Table6_DT_AE_R_C
   ⭐ TABLE 6: DT + AE + R + C signatures (ALL)

   📈 Model: COXPH
      ✅ C-Index: 0.6122 ± 0.0275
         p-value: 0.0329 ± 0.0440
         HR:      1.9149 ± 0.3843
         iAUC:    0.6723 ± 0.0407 [0.614-0.740]

📊 FINAL RESULTS SUMMARY WITH INTEGRATED AUC AND MULTIPLE MODELS

   ┌─────────────────────────────────────────┬──────────┬────────────────┬────────────┬────────────┬────────────────┐
   │ Configuration                           │ Model    │ C-Index        │ p-value    │ HR         │ iAUC        

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
FERRETTI ET AL. (2024) REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC
================================================================================

Complete reproduction with Digital Twin (3-D MLPA) integration + integrated AUC.
Now includes Neural Cox and Ensemble models for ALL configurations!

DOMAINS:
- Clinical (C): 7 features (age, gender, T/N/M stage, overall stage, histology)
- Radiomics (R): 1706 features (from data3d_1706features)
- AutoEncoder (AE): 512 features (select top 15)
- Digital Twin (DT): 6 features from 3-D MLPA simulation
    * Sim_Growth_Rate   - Growth rate from MLPA cellular automaton simulation
    * Sim_Necrosis_Ratio - Ratio of necrotic cells from simulation
    * Bio_Alpha         - Proliferation rate α = 0.01 + 0.05 * normalized_entropy
    * Bio_Necrosis      - Necrosis rate β = 0.01 + 0.08 * (1 - sphericity)
    * Radiomics_Entropy - Tumor entropy (from histogram)
    * Radiomics_Sphericity - Tumor sphericity (surface area / volume)

MODELS:
- Cox Proportional Hazards (CoxPH): Classical survival model
- Neural Cox (DeepSurv-style): Neural network for non-linear survival prediction
- Ensemble: Weighted combination of CoxPH, Neural Cox, and Gradient Boosting

METRICS:
- C-Index: Concordance index
- p-value: Log-rank test
- HR: Hazard ratio
- iAUC: Integrated time-dependent AUC

CONFIGURATIONS:
- Single domains: C, AE, R, DT
- Table 5 (Feature-Level Fusion): Concatenate features
- Table 6 (Signature-Level Fusion): Combine domain signatures
- Including DT combinations: DT+C, DT+R, DT+R+C, DT+AE+R+C

Reference: Ferretti et al. (2024) CMPB 258:108496
Author: Generated for NSCLC survival analysis with iAUC integration
================================================================================
"""

import numpy as np
import pandas as pd
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Survival analysis
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index

# Scikit-survival for integrated AUC and Gradient Boosting Survival
try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.ensemble import GradientBoostingSurvivalAnalysis
    from sksurv.linear_model import CoxnetSurvivalAnalysis
    SKSURV_AVAILABLE = True
except ImportError:
    print("⚠️  Warning: scikit-survival not installed. iAUC and some models will not be available.")
    print("   Install with: pip install scikit-survival")
    SKSURV_AVAILABLE = False

# PyTorch for Neural Cox
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    print("⚠️  Warning: PyTorch not installed. Neural Cox will use fallback.")
    print("   Install with: pip install torch")
    TORCH_AVAILABLE = False

# Machine learning
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, kendalltau

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
plt.style.use('default')

from tqdm import tqdm
import time
import pickle

# =============================================================================
# CONFIGURATION
# =============================================================================

# ============== FILE PATHS ==============
# Radiomics features (1706 features)
DATA_DIR = Path('./data3d_1706features/outputs')
RADIOMICS_NPY_FILE = DATA_DIR / 'features_normalized_standard.npy'
RADIOMICS_NAMES_JSON = DATA_DIR / 'feature_names_normalized_standard.json'
PATIENT_IDS_JSON = DATA_DIR / 'patient_ids_normalized_standard.json'

# Clinical data
CLINICAL_FILE = Path('./nsclc/NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv')

# AutoEncoder features (512 features)
DEEP_FEATURES_FILE = Path('./data_3d_ae_parallel/features/features_deep_512.csv')

# Digital Twin (3-D MLPA) features
DT_FEATURES_FILE = Path('./digital_twin_output/all_patients_biological_parameters.csv')

# Output directory
OUTPUT_DIR = Path('./ferretti_with_3d_mlpa_iauc_output')

# =============================================================================

# Digital Twin feature columns
DT_FEATURE_COLUMNS = [
    'Sim_Growth_Rate',
    'Sim_Necrosis_Ratio',
    'Bio_Alpha',
    'Bio_Necrosis',
    'Radiomics_Entropy',
    'Radiomics_Sphericity'
]

# Ferretti parameters
TOP_K_AE = 15
TOP_K_RADIOMICS = 10
CORRELATION_THRESHOLD = 0.70
LASSO_PENALIZER = 0.1
N_SPLITS = 5
RANDOM_STATE = 42

# Neural Cox parameters
NEURAL_COX_HIDDEN_LAYERS = [64, 32]  # Hidden layer sizes
NEURAL_COX_DROPOUT = 0.3
NEURAL_COX_EPOCHS = 100
NEURAL_COX_LR = 0.001
NEURAL_COX_BATCH_SIZE = 32
NEURAL_COX_PATIENCE = 10  # Early stopping patience

# Ensemble parameters
ENSEMBLE_WEIGHTS = {
    'coxph': 0.4,
    'neural_cox': 0.3,
    'gradient_boosting': 0.3
}

# Model selection: which models to run
RUN_COXPH = True
RUN_NEURAL_COX = True
RUN_ENSEMBLE = True

# Plotting
CREATE_KM_PLOTS = True
PLOT_DPI = 150

# iAUC evaluation time points (in years)
IAUC_TIME_POINTS = np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0])

# Sensitivity analysis
RUN_SENSITIVITY_ANALYSIS = True
SENSITIVITY_PERTURBATION_LEVELS = [0.1, 0.2]  # ±10%, ±20%


# =============================================================================
# NEURAL COX MODEL (DeepSurv-style)
# =============================================================================

if TORCH_AVAILABLE:
    class NeuralCoxModel(nn.Module):
        """
        Neural Cox Proportional Hazards Model (DeepSurv-style).
        
        This network outputs a risk score that is used in the Cox partial likelihood.
        Higher risk score = worse prognosis (higher hazard).
        """
        
        def __init__(self, input_dim: int, hidden_layers: List[int] = [64, 32], 
                     dropout: float = 0.3):
            super(NeuralCoxModel, self).__init__()
            
            layers = []
            prev_dim = input_dim
            
            for hidden_dim in hidden_layers:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                prev_dim = hidden_dim
            
            # Final output layer (single risk score)
            layers.append(nn.Linear(prev_dim, 1))
            
            self.network = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.network(x)
    
    
    def cox_partial_likelihood_loss(risk_scores: torch.Tensor, 
                                    times: torch.Tensor, 
                                    events: torch.Tensor) -> torch.Tensor:
        """
        Negative log partial likelihood for Cox model.
        
        Parameters:
        -----------
        risk_scores : tensor of shape (batch_size, 1)
            Predicted risk scores from the network
        times : tensor of shape (batch_size,)
            Survival times
        events : tensor of shape (batch_size,)
            Event indicators (1=event, 0=censored)
        
        Returns:
        --------
        loss : scalar tensor
            Negative log partial likelihood
        """
        risk_scores = risk_scores.squeeze()
        
        # Sort by time (descending for risk set computation)
        sorted_indices = torch.argsort(times, descending=True)
        sorted_risk = risk_scores[sorted_indices]
        sorted_events = events[sorted_indices]
        
        # Compute log risk for each subject
        max_risk = sorted_risk.max()
        log_risk = sorted_risk - max_risk
        
        # Cumulative sum of exp(risk) for risk sets (from longest to shortest survival)
        exp_risk = torch.exp(log_risk)
        cumsum_exp_risk = torch.cumsum(exp_risk, dim=0)
        
        # Log of cumulative sum
        log_cumsum = torch.log(cumsum_exp_risk + 1e-7) + max_risk
        
        # Partial likelihood: sum over events
        # L = sum_{i: event} [ risk_i - log(sum_{j in R_i} exp(risk_j)) ]
        event_mask = sorted_events == 1
        
        if event_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True)
        
        partial_likelihood = (sorted_risk[event_mask] - log_cumsum[event_mask]).sum()
        
        # Return negative (for minimization)
        return -partial_likelihood / event_mask.sum()
    
    
    class NeuralCoxTrainer:
        """Trainer for Neural Cox model with early stopping."""
        
        def __init__(self, input_dim: int, 
                     hidden_layers: List[int] = [64, 32],
                     dropout: float = 0.3,
                     lr: float = 0.001,
                     epochs: int = 100,
                     batch_size: int = 32,
                     patience: int = 10,
                     device: str = None):
            
            self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
            self.model = NeuralCoxModel(input_dim, hidden_layers, dropout).to(self.device)
            self.optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)
            self.epochs = epochs
            self.batch_size = batch_size
            self.patience = patience
            self.best_loss = float('inf')
            self.best_state = None
            self.patience_counter = 0
        
        def fit(self, X_train: np.ndarray, time_train: np.ndarray, event_train: np.ndarray,
                X_val: np.ndarray = None, time_val: np.ndarray = None, event_val: np.ndarray = None):
            """Train the Neural Cox model."""
            
            # Convert to tensors
            X_tensor = torch.FloatTensor(X_train).to(self.device)
            time_tensor = torch.FloatTensor(time_train).to(self.device)
            event_tensor = torch.FloatTensor(event_train).to(self.device)
            
            # Create dataset
            dataset = TensorDataset(X_tensor, time_tensor, event_tensor)
            dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
            
            # Validation data
            if X_val is not None:
                X_val_tensor = torch.FloatTensor(X_val).to(self.device)
                time_val_tensor = torch.FloatTensor(time_val).to(self.device)
                event_val_tensor = torch.FloatTensor(event_val).to(self.device)
            
            self.model.train()
            
            for epoch in range(self.epochs):
                epoch_loss = 0.0
                n_batches = 0
                
                for batch_X, batch_time, batch_event in dataloader:
                    self.optimizer.zero_grad()
                    
                    risk_scores = self.model(batch_X)
                    loss = cox_partial_likelihood_loss(risk_scores, batch_time, batch_event)
                    
                    if torch.isnan(loss) or torch.isinf(loss):
                        continue
                    
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                    self.optimizer.step()
                    
                    epoch_loss += loss.item()
                    n_batches += 1
                
                # Validation and early stopping
                if X_val is not None and n_batches > 0:
                    self.model.eval()
                    with torch.no_grad():
                        val_risk = self.model(X_val_tensor)
                        val_loss = cox_partial_likelihood_loss(val_risk, time_val_tensor, event_val_tensor)
                    
                    if val_loss < self.best_loss:
                        self.best_loss = val_loss
                        self.best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
                        self.patience_counter = 0
                    else:
                        self.patience_counter += 1
                    
                    if self.patience_counter >= self.patience:
                        break
                    
                    self.model.train()
            
            # Load best model
            if self.best_state is not None:
                self.model.load_state_dict(self.best_state)
        
        def predict_risk(self, X: np.ndarray) -> np.ndarray:
            """Predict risk scores for new data."""
            self.model.eval()
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X).to(self.device)
                risk_scores = self.model(X_tensor).cpu().numpy().flatten()
            return risk_scores


# =============================================================================
# GRADIENT BOOSTING SURVIVAL MODEL (from scikit-survival)
# =============================================================================

class GradientBoostingSurvival:
    """Wrapper for scikit-survival's Gradient Boosting Survival Analysis."""
    
    def __init__(self, n_estimators: int = 100, learning_rate: float = 0.1,
                 max_depth: int = 3, min_samples_split: int = 10,
                 random_state: int = 42):
        
        if not SKSURV_AVAILABLE:
            raise ImportError("scikit-survival is required for GradientBoostingSurvival")
        
        self.model = GradientBoostingSurvivalAnalysis(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=random_state,
            subsample=0.8
        )
    
    def fit(self, X: np.ndarray, time: np.ndarray, event: np.ndarray):
        """Fit the gradient boosting model."""
        # Create structured array for scikit-survival
        y = np.array([(bool(e), t) for e, t in zip(event, time)],
                     dtype=[('event', bool), ('time', float)])
        self.model.fit(X, y)
        return self
    
    def predict_risk(self, X: np.ndarray) -> np.ndarray:
        """Predict risk scores (higher = worse prognosis)."""
        return self.model.predict(X)


# =============================================================================
# ENSEMBLE MODEL
# =============================================================================

class EnsembleSurvivalModel:
    """
    Ensemble model combining CoxPH, Neural Cox, and Gradient Boosting.
    
    The ensemble averages normalized risk scores from multiple models
    using specified weights.
    """
    
    def __init__(self, weights: Dict[str, float] = None, use_neural_cox: bool = True,
                 use_gradient_boosting: bool = True):
        
        self.weights = weights or {'coxph': 0.4, 'neural_cox': 0.3, 'gradient_boosting': 0.3}
        self.use_neural_cox = use_neural_cox and TORCH_AVAILABLE
        self.use_gradient_boosting = use_gradient_boosting and SKSURV_AVAILABLE
        
        # Normalize weights based on available models
        self._normalize_weights()
        
        self.coxph_model = None
        self.neural_cox_trainer = None
        self.gb_model = None
        self.scaler = StandardScaler()
        self.coxph_features = None
    
    def _normalize_weights(self):
        """Normalize weights to sum to 1 based on available models."""
        active_weights = {'coxph': self.weights.get('coxph', 0.4)}
        
        if self.use_neural_cox:
            active_weights['neural_cox'] = self.weights.get('neural_cox', 0.3)
        if self.use_gradient_boosting:
            active_weights['gradient_boosting'] = self.weights.get('gradient_boosting', 0.3)
        
        total = sum(active_weights.values())
        self.active_weights = {k: v/total for k, v in active_weights.items()}
    
    def fit(self, X_train: np.ndarray, time_train: np.ndarray, event_train: np.ndarray,
            feature_names: List[str] = None):
        """Fit all component models."""
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X_train)
        
        # 1. Fit CoxPH
        try:
            self.coxph_features = feature_names or [f'F{i}' for i in range(X_scaled.shape[1])]
            df_train = pd.DataFrame(X_scaled, columns=self.coxph_features)
            df_train['time_years'] = time_train
            df_train['event'] = event_train
            
            self.coxph_model = CoxPHFitter(penalizer=LASSO_PENALIZER)
            feature_cols = [c for c in df_train.columns if c not in ['time_years', 'event']]
            self.coxph_model.fit(df_train[feature_cols + ['time_years', 'event']],
                                 duration_col='time_years', event_col='event')
        except Exception as e:
            print(f"      ⚠️  CoxPH fitting failed: {str(e)[:50]}")
            self.coxph_model = None
        
        # 2. Fit Neural Cox
        if self.use_neural_cox:
            try:
                self.neural_cox_trainer = NeuralCoxTrainer(
                    input_dim=X_scaled.shape[1],
                    hidden_layers=NEURAL_COX_HIDDEN_LAYERS,
                    dropout=NEURAL_COX_DROPOUT,
                    lr=NEURAL_COX_LR,
                    epochs=NEURAL_COX_EPOCHS,
                    batch_size=NEURAL_COX_BATCH_SIZE,
                    patience=NEURAL_COX_PATIENCE
                )
                
                # Use 20% of training data for validation within Neural Cox
                n_val = max(1, int(len(X_scaled) * 0.2))
                indices = np.random.permutation(len(X_scaled))
                val_idx, train_idx = indices[:n_val], indices[n_val:]
                
                self.neural_cox_trainer.fit(
                    X_scaled[train_idx], time_train[train_idx], event_train[train_idx],
                    X_scaled[val_idx], time_train[val_idx], event_train[val_idx]
                )
            except Exception as e:
                print(f"      ⚠️  Neural Cox fitting failed: {str(e)[:50]}")
                self.neural_cox_trainer = None
        
        # 3. Fit Gradient Boosting
        if self.use_gradient_boosting:
            try:
                self.gb_model = GradientBoostingSurvival(
                    n_estimators=100,
                    learning_rate=0.1,
                    max_depth=3,
                    random_state=RANDOM_STATE
                )
                self.gb_model.fit(X_scaled, time_train, event_train)
            except Exception as e:
                print(f"      ⚠️  Gradient Boosting fitting failed: {str(e)[:50]}")
                self.gb_model = None
        
        return self
    
    def predict_risk(self, X: np.ndarray) -> np.ndarray:
        """Predict ensemble risk scores."""
        
        X_scaled = self.scaler.transform(X)
        
        risk_scores = []
        weights = []
        
        # CoxPH predictions
        if self.coxph_model is not None:
            try:
                df_pred = pd.DataFrame(X_scaled, columns=self.coxph_features)
                coxph_risk = self.coxph_model.predict_partial_hazard(df_pred).values.flatten()
                # Normalize to [0, 1]
                if coxph_risk.max() - coxph_risk.min() > 1e-8:
                    coxph_risk = (coxph_risk - coxph_risk.min()) / (coxph_risk.max() - coxph_risk.min())
                risk_scores.append(coxph_risk)
                weights.append(self.active_weights.get('coxph', 0))
            except:
                pass
        
        # Neural Cox predictions
        if self.neural_cox_trainer is not None:
            try:
                neural_risk = self.neural_cox_trainer.predict_risk(X_scaled)
                # Normalize to [0, 1]
                if neural_risk.max() - neural_risk.min() > 1e-8:
                    neural_risk = (neural_risk - neural_risk.min()) / (neural_risk.max() - neural_risk.min())
                risk_scores.append(neural_risk)
                weights.append(self.active_weights.get('neural_cox', 0))
            except:
                pass
        
        # Gradient Boosting predictions
        if self.gb_model is not None:
            try:
                gb_risk = self.gb_model.predict_risk(X_scaled)
                # Normalize to [0, 1]
                if gb_risk.max() - gb_risk.min() > 1e-8:
                    gb_risk = (gb_risk - gb_risk.min()) / (gb_risk.max() - gb_risk.min())
                risk_scores.append(gb_risk)
                weights.append(self.active_weights.get('gradient_boosting', 0))
            except:
                pass
        
        if len(risk_scores) == 0:
            return np.zeros(len(X))
        
        # Weighted average
        weights = np.array(weights)
        weights = weights / weights.sum()  # Renormalize
        
        ensemble_risk = np.zeros(len(X))
        for risk, w in zip(risk_scores, weights):
            ensemble_risk += w * risk
        
        return ensemble_risk


# =============================================================================
# DATA LOADING
# =============================================================================

def load_clinical_data(clinical_file: Path) -> pd.DataFrame:
    """Load clinical data with 7 features (Ferretti method)."""
    
    print("\n" + "="*100)
    print("📊 LOADING CLINICAL DATA")
    print("="*100)
    
    df = pd.read_csv(clinical_file)
    df.columns = [c.strip() for c in df.columns]
    
    print(f"\n✅ Loaded {len(df)} patients")
    
    # Standardize column names
    col_map = {}
    for col in df.columns:
        col_lower = col.lower()
        if 'patient' in col_lower:
            col_map[col] = 'PatientID'
        elif col_lower == 'age':
            col_map[col] = 'age'
        elif 't.stage' in col_lower or 't stage' in col_lower:
            col_map[col] = 'T_Stage'
        elif 'n.stage' in col_lower or 'n stage' in col_lower:
            col_map[col] = 'N_Stage'
        elif 'm.stage' in col_lower or 'm stage' in col_lower:
            col_map[col] = 'M_Stage'
        elif 'overall' in col_lower and 'stage' in col_lower:
            col_map[col] = 'Overall_Stage'
        elif 'histology' in col_lower:
            col_map[col] = 'Histology'
        elif 'gender' in col_lower or 'sex' in col_lower:
            col_map[col] = 'gender'
        elif 'survival' in col_lower and 'time' in col_lower:
            col_map[col] = 'Survival_time'
        elif 'dead' in col_lower or 'event' in col_lower or 'status' in col_lower:
            col_map[col] = 'event'
    
    df = df.rename(columns=col_map)
    df['PatientID'] = df['PatientID'].astype(str)
    
    # Survival time
    df['Survival_time'] = pd.to_numeric(df['Survival_time'], errors='coerce')
    df['event'] = pd.to_numeric(df['event'], errors='coerce')
    df['time_years'] = df['Survival_time'] / 365.25
    
    print(f"   Time range: {df['time_years'].min():.2f} - {df['time_years'].max():.2f} years")
    print(f"   Events: {df['event'].sum():.0f}/{len(df)} ({100*df['event'].mean():.1f}%)")
    
    # Missing data imputation
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['age'].fillna(df['age'].median(), inplace=True)
    
    for stage_col in ['T_Stage', 'N_Stage', 'M_Stage']:
        if stage_col in df.columns:
            df[stage_col] = pd.to_numeric(df[stage_col], errors='coerce')
            mode_val = df[stage_col].mode()[0] if len(df[stage_col].mode()) > 0 else 0
            df[stage_col].fillna(mode_val, inplace=True)
        else:
            df[stage_col] = 0
    
    if 'Overall_Stage' in df.columns:
        mode_val = df['Overall_Stage'].mode()[0] if len(df['Overall_Stage'].mode()) > 0 else 'IIIb'
        df['Overall_Stage'].fillna(mode_val, inplace=True)
    else:
        df['Overall_Stage'] = 'IIIb'
    
    if 'Histology' in df.columns:
        mode_val = df['Histology'].mode()[0] if len(df['Histology'].mode()) > 0 else 'nos'
        df['Histology'].fillna(mode_val, inplace=True)
    else:
        df['Histology'] = 'nos'
    
    if 'gender' in df.columns:
        mode_val = df['gender'].mode()[0] if len(df['gender'].mode()) > 0 else 'male'
        df['gender'].fillna(mode_val, inplace=True)
    else:
        df['gender'] = 'male'
    
    # Encode categorical variables
    df['gender_encoded'] = df['gender'].astype(str).str.lower().str.strip().map({
        'male': 1, 'female': 0, 'm': 1, 'f': 0
    }).fillna(1)
    
    df['T_Stage_encoded'] = pd.to_numeric(df['T_Stage'], errors='coerce').fillna(0)
    df['N_Stage_encoded'] = pd.to_numeric(df['N_Stage'], errors='coerce').fillna(0)
    df['M_Stage_encoded'] = pd.to_numeric(df['M_Stage'], errors='coerce').fillna(0)
    
    stage_map = {
        'i': 1, 'ia': 1, 'ib': 1,
        'ii': 2, 'iia': 2, 'iib': 2,
        'iii': 3, 'iiia': 3, 'iiib': 4,
        'iv': 5, 'iva': 5, 'ivb': 5
    }
    df['Overall_Stage_encoded'] = df['Overall_Stage'].astype(str).str.lower().str.strip().map(stage_map).fillna(0)
    
    histology_map = {}
    for val in df['Histology'].unique():
        val_str = str(val).lower().strip()
        if 'large' in val_str or 'lcc' in val_str:
            histology_map[val] = 0
        elif 'squamous' in val_str or 'scc' in val_str:
            histology_map[val] = 1
        elif 'adeno' in val_str:
            histology_map[val] = 2
        else:
            histology_map[val] = 3
    
    df['Histology_encoded'] = df['Histology'].map(histology_map).fillna(3)
    
    df = df.dropna(subset=['time_years', 'event'])
    
    print(f"\n✅ Clinical data prepared: {len(df)} patients with 7 features")
    
    return df


def load_radiomics_from_npy(npy_file: Path, 
                           names_file: Path,
                           ids_file: Path) -> Tuple[pd.DataFrame, List[str], List[str]]:
    """Load radiomics features from .npy files (1706 features)."""
    
    print("\n" + "="*100)
    print("📊 LOADING RADIOMICS FEATURES (1706 features)")
    print("="*100)
    
    print(f"\n📂 Loading: {npy_file}")
    if not npy_file.exists():
        raise FileNotFoundError(f"Radiomics file not found: {npy_file}")
    
    radiomics_array = np.load(npy_file)
    print(f"   ✅ Shape: {radiomics_array.shape}")
    
    with open(names_file, 'r') as f:
        feature_names = json.load(f)
    
    with open(ids_file, 'r') as f:
        patient_ids = json.load(f)
    
    df_radiomics = pd.DataFrame(radiomics_array, columns=feature_names)
    df_radiomics['PatientID'] = patient_ids
    
    cols = ['PatientID'] + [c for c in df_radiomics.columns if c != 'PatientID']
    df_radiomics = df_radiomics[cols]
    
    print(f"   ✅ Patients: {len(patient_ids)}, Features: {len(feature_names)}")
    
    return df_radiomics, feature_names, patient_ids


def load_ae_features(ae_file: Path) -> Tuple[pd.DataFrame, List[str]]:
    """Load AutoEncoder features (512 features)."""
    
    print("\n" + "="*100)
    print("📊 LOADING AUTOENCODER FEATURES (512 features)")
    print("="*100)
    
    print(f"\n📂 Loading: {ae_file}")
    df_ae = pd.read_csv(ae_file)
    
    ae_features = [c for c in df_ae.columns if 'Deep_F' in c or 'deep_f' in c.lower()]
    
    print(f"   ✅ Patients: {len(df_ae)}, Features: {len(ae_features)}")
    
    return df_ae, ae_features


def load_digital_twin_features(dt_file: Path) -> Tuple[pd.DataFrame, List[str]]:
    """Load Digital Twin (3-D MLPA) features."""
    
    print("\n" + "="*100)
    print("📊 LOADING DIGITAL TWIN (3-D MLPA) FEATURES")
    print("="*100)
    
    print(f"\n📂 Loading: {dt_file}")
    
    if not dt_file.exists():
        print(f"   ⚠️  Digital Twin file not found: {dt_file}")
        print(f"   Please run digital_twin_pipeline.py first to generate features.")
        return pd.DataFrame({'PatientID': []}), []
    
    df_dt = pd.read_csv(dt_file)
    df_dt['PatientID'] = df_dt['PatientID'].astype(str)
    
    available_dt_features = [f for f in DT_FEATURE_COLUMNS if f in df_dt.columns]
    missing_features = [f for f in DT_FEATURE_COLUMNS if f not in df_dt.columns]
    
    if missing_features:
        print(f"   ⚠️  Missing features: {missing_features}")
    
    print(f"   ✅ Patients: {len(df_dt)}")
    print(f"   ✅ Available DT features: {len(available_dt_features)}")
    
    print(f"\n   📊 3-D MLPA Feature Statistics:")
    print(f"   {'─'*70}")
    print(f"   {'Feature':<25} {'Mean':>12} {'Std':>12} {'Min':>12} {'Max':>12}")
    print(f"   {'─'*70}")
    
    for feat in available_dt_features:
        values = df_dt[feat]
        print(f"   {feat:<25} {values.mean():>12.6f} {values.std():>12.6f} {values.min():>12.6f} {values.max():>12.6f}")
    
    print(f"   {'─'*70}")
    
    return df_dt, available_dt_features


# =============================================================================
# FEATURE SELECTION (FERRETTI PIPELINE)
# =============================================================================

def remove_correlated_features(df: pd.DataFrame, 
                               features: List[str], 
                               threshold: float = 0.70) -> List[str]:
    """Remove highly correlated features (Ferretti method)."""
    
    if len(features) <= 1:
        return features
    
    try:
        correlation_matrix = df[features].corr().abs()
        features_to_remove = set()
        
        for i in range(len(correlation_matrix.columns)):
            for j in range(i + 1, len(correlation_matrix.columns)):
                if correlation_matrix.iloc[i, j] >= threshold:
                    avg_corr_i = correlation_matrix.iloc[i, :].mean()
                    avg_corr_j = correlation_matrix.iloc[j, :].mean()
                    
                    if avg_corr_i > avg_corr_j:
                        features_to_remove.add(correlation_matrix.columns[i])
                    else:
                        features_to_remove.add(correlation_matrix.columns[j])
        
        selected_features = [f for f in features if f not in features_to_remove]
        return selected_features
        
    except Exception as e:
        print(f"   ⚠️  Error in correlation filtering: {str(e)[:100]}")
        return features


def remove_constant_features(df: pd.DataFrame, features: List[str]) -> List[str]:
    """Remove features with zero variance."""
    
    if len(features) == 0:
        return features
    
    try:
        variances = df[features].var()
        non_constant = variances[variances > 0].index.tolist()
        return non_constant
    except:
        return features


def select_features_univariate_cox(train_df: pd.DataFrame, 
                                   features: List[str], 
                                   top_k: int) -> List[str]:
    """Select top K features using univariate Cox PH (by p-value)."""
    
    if len(features) <= top_k:
        return features
    
    p_values = []
    
    for f in features:
        try:
            cph = CoxPHFitter()
            subset = train_df[[f, 'time_years', 'event']].copy()
            cph.fit(subset, duration_col='time_years', event_col='event')
            p_values.append((f, cph.summary.loc[f, 'p']))
        except:
            p_values.append((f, 1.0))
    
    p_values.sort(key=lambda x: x[1])
    selected = [x[0] for x in p_values[:top_k]]
    
    return selected


# =============================================================================
# SIGNATURE GENERATION (TABLE 6 APPROACH)
# =============================================================================

def generate_domain_signature(train_df: pd.DataFrame, 
                              val_df: pd.DataFrame,
                              features: List[str], 
                              domain_name: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """Generate domain-specific signature (risk scores) using CoxPH."""
    
    if len(features) == 0:
        return None, None
    
    try:
        scaler = StandardScaler()
        train_scaled = train_df.copy()
        val_scaled = val_df.copy()
        train_scaled[features] = scaler.fit_transform(train_df[features])
        val_scaled[features] = scaler.transform(val_df[features])
        
        cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
        cph.fit(train_scaled[features + ['time_years', 'event']],
               duration_col='time_years', event_col='event')
        
        train_signature = cph.predict_partial_hazard(train_scaled[features]).values.flatten()
        val_signature = cph.predict_partial_hazard(val_scaled[features]).values.flatten()
        
        return train_signature, val_signature
        
    except Exception as e:
        print(f"   ⚠️  Error generating {domain_name} signature: {str(e)[:50]}")
        return None, None


# =============================================================================
# SURVIVAL METRICS WITH INTEGRATED AUC
# =============================================================================

def calculate_all_metrics(time_test: np.ndarray, 
                         event_test: np.ndarray,
                         risk_scores: np.ndarray,
                         time_train: np.ndarray = None,
                         event_train: np.ndarray = None) -> Dict[str, float]:
    """
    Calculate C-Index, log-rank p-value, Hazard Ratio, and integrated AUC.
    """
    
    metrics = {}
    
    # 1. C-Index (Concordance Index)
    try:
        c_index = concordance_index(time_test, -risk_scores, event_test)
        metrics['c_index'] = c_index
    except:
        metrics['c_index'] = np.nan
    
    # 2. Log-rank p-value (stratified by median risk)
    try:
        median_risk = np.median(risk_scores)
        group = (risk_scores >= median_risk).astype(int)
        
        results = logrank_test(
            time_test[group == 0], time_test[group == 1],
            event_test[group == 0], event_test[group == 1]
        )
        metrics['p_value'] = results.p_value
    except:
        metrics['p_value'] = np.nan
    
    # 3. Hazard Ratio (high risk vs low risk)
    try:
        median_risk = np.median(risk_scores)
        group = (risk_scores >= median_risk).astype(int)
        
        df_temp = pd.DataFrame({
            'time': time_test,
            'event': event_test,
            'risk_group': group
        })
        
        cph = CoxPHFitter()
        cph.fit(df_temp, duration_col='time', event_col='event')
        hr = np.exp(cph.params_['risk_group'])
        metrics['hazard_ratio'] = hr
    except:
        metrics['hazard_ratio'] = np.nan
    
    # 4. Integrated AUC (iAUC) - requires scikit-survival
    if SKSURV_AVAILABLE and time_train is not None and event_train is not None:
        try:
            # Create structured arrays for scikit-survival
            train_y = np.array(
                [(bool(e), t) for e, t in zip(event_train, time_train)],
                dtype=[('event', bool), ('time', float)]
            )
            
            test_y = np.array(
                [(bool(e), t) for e, t in zip(event_test, time_test)],
                dtype=[('event', bool), ('time', float)]
            )
            
            # Define evaluation time points
            max_time = min(np.percentile(time_test[event_test == 1], 90), 2.0)
            times = IAUC_TIME_POINTS[IAUC_TIME_POINTS <= max_time]
            
            min_time = max(0.1, np.percentile(time_test[event_test == 1], 10))
            times = times[times >= min_time]
            
            if len(times) >= 2:
                auc_scores, mean_auc = cumulative_dynamic_auc(
                    train_y, test_y, risk_scores, times
                )
                
                metrics['iauc'] = mean_auc
                metrics['auc_by_time'] = dict(zip(times, auc_scores))
                metrics['iauc_min'] = np.min(auc_scores)
                metrics['iauc_max'] = np.max(auc_scores)
            else:
                metrics['iauc'] = np.nan
                metrics['auc_by_time'] = {}
                metrics['iauc_min'] = np.nan
                metrics['iauc_max'] = np.nan
                
        except Exception as e:
            metrics['iauc'] = np.nan
            metrics['auc_by_time'] = {}
            metrics['iauc_min'] = np.nan
            metrics['iauc_max'] = np.nan
    else:
        metrics['iauc'] = np.nan
        metrics['auc_by_time'] = {}
        metrics['iauc_min'] = np.nan
        metrics['iauc_max'] = np.nan
    
    return metrics


# =============================================================================
# KAPLAN-MEIER PLOTTING
# =============================================================================

def plot_km_curve(time: np.ndarray, 
                  event: np.ndarray, 
                  risk_scores: np.ndarray,
                  title: str, 
                  output_path: Path, 
                  dataset_type: str = "Validation"):
    """Plot Kaplan-Meier curve (high risk vs low risk)."""
    
    try:
        median_risk = np.median(risk_scores)
        high_risk_mask = risk_scores >= median_risk
        
        kmf_high = KaplanMeierFitter()
        kmf_low = KaplanMeierFitter()
        
        kmf_high.fit(time[high_risk_mask], event[high_risk_mask], label='High Risk')
        kmf_low.fit(time[~high_risk_mask], event[~high_risk_mask], label='Low Risk')
        
        results = logrank_test(
            time[high_risk_mask], time[~high_risk_mask],
            event[high_risk_mask], event[~high_risk_mask]
        )
        
        fig, ax = plt.subplots(figsize=(10, 7))
        
        kmf_high.plot_survival_function(ax=ax, ci_show=True, color='red', linewidth=2.5, alpha=0.8)
        kmf_low.plot_survival_function(ax=ax, ci_show=True, color='blue', linewidth=2.5, alpha=0.8)
        
        significance = '✅ Significant' if results.p_value < 0.05 else '❌ Not Significant'
        ax.set_title(f'{title} ({dataset_type})\nLog-Rank p = {results.p_value:.6f} ({significance})',
                    fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Years', fontsize=12, fontweight='bold')
        ax.set_ylabel('Survival Probability', fontsize=12, fontweight='bold')
        ax.legend(fontsize=11, loc='best', framealpha=0.9)
        ax.grid(alpha=0.3, linestyle='--')
        ax.set_ylim([0, 1.05])
        
        n_high = high_risk_mask.sum()
        n_low = (~high_risk_mask).sum()
        events_high = event[high_risk_mask].sum()
        events_low = event[~high_risk_mask].sum()
        
        stats_text = (f'High Risk: n={n_high}, events={int(events_high)}\n'
                     f'Low Risk: n={n_low}, events={int(events_low)}')
        
        ax.text(0.02, 0.02, stats_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='bottom',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        plt.tight_layout()
        plt.savefig(output_path, dpi=PLOT_DPI, bbox_inches='tight')
        plt.close()
        
        return results.p_value
        
    except Exception as e:
        print(f"   ⚠️  Error plotting KM curve: {str(e)[:50]}")
        return np.nan


# =============================================================================
# TRAIN MODEL HELPER FUNCTION
# =============================================================================

def train_and_predict(model_type: str,
                      X_train: np.ndarray,
                      X_val: np.ndarray,
                      time_train: np.ndarray,
                      event_train: np.ndarray,
                      feature_names: List[str] = None) -> Optional[np.ndarray]:
    """
    Train a model and return predictions.
    
    Parameters:
    -----------
    model_type : str
        One of 'coxph', 'neural_cox', 'ensemble'
    X_train : ndarray
        Training features (already scaled)
    X_val : ndarray
        Validation features (already scaled)
    time_train : ndarray
        Training survival times
    event_train : ndarray
        Training event indicators
    feature_names : list
        List of feature column names
    
    Returns:
    --------
    risk_scores : array or None
        Predicted risk scores for validation set
    """
    
    if feature_names is None:
        feature_names = [f'F{i}' for i in range(X_train.shape[1])]
    
    # =========================
    # CoxPH Model
    # =========================
    if model_type == 'coxph':
        try:
            df_train = pd.DataFrame(X_train, columns=feature_names)
            df_train['time_years'] = time_train
            df_train['event'] = event_train
            
            df_val = pd.DataFrame(X_val, columns=feature_names)
            
            cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
            cph.fit(df_train[feature_names + ['time_years', 'event']],
                   duration_col='time_years', event_col='event')
            
            risk_scores_val = cph.predict_partial_hazard(df_val[feature_names]).values.flatten()
            return risk_scores_val
        except Exception as e:
            return None
    
    # =========================
    # Neural Cox Model
    # =========================
    elif model_type == 'neural_cox':
        if not TORCH_AVAILABLE:
            return None
        
        try:
            trainer = NeuralCoxTrainer(
                input_dim=X_train.shape[1],
                hidden_layers=NEURAL_COX_HIDDEN_LAYERS,
                dropout=NEURAL_COX_DROPOUT,
                lr=NEURAL_COX_LR,
                epochs=NEURAL_COX_EPOCHS,
                batch_size=NEURAL_COX_BATCH_SIZE,
                patience=NEURAL_COX_PATIENCE
            )
            
            # Split training data for internal validation
            n_internal_val = max(1, int(len(X_train) * 0.15))
            indices = np.random.permutation(len(X_train))
            internal_val_idx, internal_train_idx = indices[:n_internal_val], indices[n_internal_val:]
            
            trainer.fit(
                X_train[internal_train_idx],
                time_train[internal_train_idx],
                event_train[internal_train_idx],
                X_train[internal_val_idx],
                time_train[internal_val_idx],
                event_train[internal_val_idx]
            )
            
            risk_scores_val = trainer.predict_risk(X_val)
            return risk_scores_val
        except Exception as e:
            return None
    
    # =========================
    # Ensemble Model
    # =========================
    elif model_type == 'ensemble':
        try:
            ensemble = EnsembleSurvivalModel(
                weights=ENSEMBLE_WEIGHTS,
                use_neural_cox=TORCH_AVAILABLE,
                use_gradient_boosting=SKSURV_AVAILABLE
            )
            
            # Note: ensemble handles its own scaling internally
            # So we pass unscaled data and let it scale
            ensemble.fit(X_train, time_train, event_train, 
                        feature_names=feature_names)
            
            risk_scores_val = ensemble.predict_risk(X_val)
            return risk_scores_val
        except Exception as e:
            return None
    
    return None


# =============================================================================
# MAIN EXPERIMENT WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS
# =============================================================================

def run_experiment_with_3d_mlpa(df_merged: pd.DataFrame,
                                radiomics_features: List[str],
                                ae_features: List[str],
                                clinical_features: List[str],
                                dt_features: List[str],
                                n_splits: int = 5,
                                random_state: int = 42,
                                create_km_plots: bool = True) -> Dict:
    """
    Run experiment with 3-D MLPA Digital Twin features, integrated AUC,
    and multiple models (CoxPH, Neural Cox, Ensemble).
    
    ALL configurations (including Table 6) are tested with all models.
    """
    
    print("\n" + "="*100)
    print("📊 RUNNING EXPERIMENT WITH 3-D MLPA DIGITAL TWIN + iAUC + MULTIPLE MODELS")
    print("="*100)
    
    print("\n🔧 MODEL AVAILABILITY:")
    print(f"   • CoxPH:            ✅ Available (lifelines)")
    print(f"   • Neural Cox:       {'✅ Available (PyTorch)' if TORCH_AVAILABLE else '❌ Not available (install PyTorch)'}")
    print(f"   • Gradient Boosting: {'✅ Available (scikit-survival)' if SKSURV_AVAILABLE else '❌ Not available'}")
    print(f"   • Ensemble:         {'✅ Available' if (TORCH_AVAILABLE or SKSURV_AVAILABLE) else '⚠️ Limited (CoxPH only)'}")
    print(f"   • iAUC:             {'✅ Available' if SKSURV_AVAILABLE else '❌ Not available'}")
    
    # Determine which models to run
    models_to_run = []
    if RUN_COXPH:
        models_to_run.append('coxph')
    if RUN_NEURAL_COX and TORCH_AVAILABLE:
        models_to_run.append('neural_cox')
    if RUN_ENSEMBLE:
        models_to_run.append('ensemble')
    
    print(f"\n🔬 MODELS TO RUN: {models_to_run}")
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Check if DT features are available
    has_dt = len(dt_features) > 0
    
    if not has_dt:
        print("\n⚠️  No Digital Twin features available. Skipping DT configurations.")
    else:
        print(f"\n✅ Digital Twin features available ({len(dt_features)})")
    
    # Define all configurations (feature combinations)
    configs = {}
    
    # ========== SINGLE DOMAINS ==========
    configs['C_Only'] = {
        'description': f'Clinical only ({len(clinical_features)} features)',
        'fusion_type': None,
        'features': clinical_features,
        'use_selection': False
    }
    
    configs['AE_Only'] = {
        'description': f'AutoEncoder (top {TOP_K_AE} from {len(ae_features)})',
        'fusion_type': None,
        'features': ae_features,
        'use_selection': True,
        'top_k': TOP_K_AE
    }
    
    configs['R_Only'] = {
        'description': f'Radiomics (top {TOP_K_RADIOMICS} from {len(radiomics_features)})',
        'fusion_type': None,
        'features': radiomics_features,
        'use_selection': True,
        'top_k': TOP_K_RADIOMICS
    }
    
    if has_dt:
        configs['DT_Only'] = {
            'description': f'Digital Twin 3-D MLPA ({len(dt_features)} features)',
            'fusion_type': None,
            'features': dt_features,
            'use_selection': False
        }
    
    # ========== TABLE 5: FEATURE-LEVEL FUSION ==========
    configs['Table5_AE_R'] = {
        'description': '📊 TABLE 5: AE + Radiomics',
        'fusion_type': 'features',
        'domains': ['AE', 'R']
    }
    
    configs['Table5_AE_C'] = {
        'description': '📊 TABLE 5: AE + Clinical',
        'fusion_type': 'features',
        'domains': ['AE', 'C']
    }
    
    configs['Table5_R_C'] = {
        'description': '📊 TABLE 5: Radiomics + Clinical',
        'fusion_type': 'features',
        'domains': ['R', 'C']
    }
    
    configs['Table5_AE_R_C'] = {
        'description': '📊 TABLE 5: AE + Radiomics + Clinical',
        'fusion_type': 'features',
        'domains': ['AE', 'R', 'C']
    }
    
    if has_dt:
        configs['Table5_DT_C'] = {
            'description': '📊 TABLE 5: DT + Clinical',
            'fusion_type': 'features',
            'domains': ['DT', 'C']
        }
        
        configs['Table5_DT_R'] = {
            'description': '📊 TABLE 5: DT + Radiomics',
            'fusion_type': 'features',
            'domains': ['DT', 'R']
        }
        
        configs['Table5_DT_R_C'] = {
            'description': '📊 TABLE 5: DT + Radiomics + Clinical',
            'fusion_type': 'features',
            'domains': ['DT', 'R', 'C']
        }
        
        configs['Table5_DT_AE_R_C'] = {
            'description': '📊 TABLE 5: DT + AE + Radiomics + Clinical (ALL)',
            'fusion_type': 'features',
            'domains': ['DT', 'AE', 'R', 'C']
        }
    
    # ========== TABLE 6: SIGNATURE-LEVEL FUSION ==========
    # Now supports ALL models (CoxPH, Neural Cox, Ensemble)
    configs['Table6_AE_R'] = {
        'description': '⭐ TABLE 6: AE + Radiomics signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'R']
    }
    
    configs['Table6_AE_C'] = {
        'description': '⭐ TABLE 6: AE + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'C']
    }
    
    configs['Table6_R_C'] = {
        'description': '⭐ TABLE 6: Radiomics + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['R', 'C']
    }
    
    configs['Table6_AE_R_C'] = {
        'description': '⭐ TABLE 6: AE + Radiomics + Clinical signatures',
        'fusion_type': 'signatures',
        'domains': ['AE', 'R', 'C']
    }
    
    if has_dt:
        configs['Table6_DT_C'] = {
            'description': '⭐ TABLE 6: DT + Clinical signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'C']
        }
        
        configs['Table6_DT_R'] = {
            'description': '⭐ TABLE 6: DT + Radiomics signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'R']
        }
        
        configs['Table6_DT_R_C'] = {
            'description': '⭐ TABLE 6: DT + Radiomics + Clinical signatures',
            'fusion_type': 'signatures',
            'domains': ['DT', 'R', 'C']
        }
        
        configs['Table6_DT_AE_R_C'] = {
            'description': '⭐ TABLE 6: DT + AE + R + C signatures (ALL)',
            'fusion_type': 'signatures',
            'domains': ['DT', 'AE', 'R', 'C']
        }
    
    # Domain feature mapping
    domain_features_map = {
        'C': clinical_features,
        'AE': ae_features,
        'R': radiomics_features,
        'DT': dt_features
    }
    
    domain_top_k_map = {
        'C': None,
        'AE': TOP_K_AE,
        'R': TOP_K_RADIOMICS,
        'DT': None
    }
    
    results = {}
    
    # Create KM plots directory
    if create_km_plots:
        km_dir = OUTPUT_DIR / "km_plots"
        km_dir.mkdir(parents=True, exist_ok=True)
    
    # Run each configuration with each model
    for config_name, config in configs.items():
        print(f"\n{'─' * 100}")
        print(f"🔬 Configuration: {config_name}")
        print(f"   {config['description']}")
        
        fusion_type = config.get('fusion_type', None)
        
        # ALL configurations now use ALL models
        models_for_config = models_to_run
        
        for model_type in models_for_config:
            result_key = f"{config_name}_{model_type}"
            
            print(f"\n   📈 Model: {model_type.upper()}")
            
            fold_metrics = {
                'c_index': [],
                'p_value': [],
                'hazard_ratio': [],
                'iauc': [],
                'iauc_min': [],
                'iauc_max': []
            }
            
            first_fold_data = None
            
            # Cross-validation
            for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_merged, df_merged['event'].astype(int)), 1):
                fold_train = df_merged.iloc[train_idx].copy()
                fold_val = df_merged.iloc[val_idx].copy()
                
                time_train = fold_train['time_years'].values
                event_train = fold_train['event'].values
                time_val = fold_val['time_years'].values
                event_val = fold_val['event'].values
                
                risk_scores_val = None
                
                # ========== TABLE 5: FEATURE-LEVEL FUSION ==========
                if fusion_type == 'features':
                    domains = config['domains']
                    fold_features = []
                    
                    for domain in domains:
                        domain_feats = domain_features_map[domain]
                        top_k = domain_top_k_map[domain]
                        
                        if top_k is not None and len(domain_feats) > top_k:
                            selected = select_features_univariate_cox(fold_train, domain_feats, top_k)
                        else:
                            selected = domain_feats
                        
                        fold_features.extend(selected)
                    
                    # Remove duplicates
                    seen = set()
                    fold_features = [f for f in fold_features if not (f in seen or seen.add(f))]
                    
                    if len(fold_features) == 0:
                        continue
                    
                    # Scale features
                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(fold_train[fold_features].values)
                    X_val = scaler.transform(fold_val[fold_features].values)
                    
                    # Train model and get predictions
                    risk_scores_val = train_and_predict(
                        model_type, X_train, X_val, time_train, event_train, fold_features
                    )
                
                # ========== TABLE 6: SIGNATURE-LEVEL FUSION ==========
                elif fusion_type == 'signatures':
                    domains = config['domains']
                    signatures_train = []
                    signatures_val = []
                    
                    for domain in domains:
                        domain_feats = domain_features_map[domain]
                        top_k = domain_top_k_map[domain]
                        
                        if top_k is not None and len(domain_feats) > top_k:
                            selected = select_features_univariate_cox(fold_train, domain_feats, top_k)
                        else:
                            selected = domain_feats
                        
                        if len(selected) == 0:
                            continue
                        
                        # Generate CoxPH-based signatures for each domain
                        sig_train, sig_val = generate_domain_signature(
                            fold_train, fold_val, selected, domain
                        )
                        
                        if sig_train is not None:
                            signatures_train.append(sig_train)
                            signatures_val.append(sig_val)
                    
                    if len(signatures_train) == 0:
                        continue
                    
                    # Stack signatures as features
                    X_train_sigs = np.column_stack(signatures_train)
                    X_val_sigs = np.column_stack(signatures_val)
                    
                    sig_names = [f'{domains[i]}_Sig' for i in range(len(signatures_train))]
                    
                    # Scale signatures
                    scaler = StandardScaler()
                    X_train_sigs_scaled = scaler.fit_transform(X_train_sigs)
                    X_val_sigs_scaled = scaler.transform(X_val_sigs)
                    
                    # Train model on signatures and get predictions
                    risk_scores_val = train_and_predict(
                        model_type, X_train_sigs_scaled, X_val_sigs_scaled, 
                        time_train, event_train, sig_names
                    )
                
                # ========== SINGLE DOMAIN ==========
                else:
                    fold_features = config['features']
                    
                    if config.get('use_selection', False) and 'top_k' in config:
                        fold_features = select_features_univariate_cox(
                            fold_train, fold_features, config['top_k']
                        )
                    
                    if len(fold_features) == 0:
                        continue
                    
                    # Scale features
                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(fold_train[fold_features].values)
                    X_val = scaler.transform(fold_val[fold_features].values)
                    
                    # Train model and get predictions
                    risk_scores_val = train_and_predict(
                        model_type, X_train, X_val, time_train, event_train, fold_features
                    )
                
                if risk_scores_val is None:
                    continue
                
                # Calculate metrics including iAUC
                metrics = calculate_all_metrics(
                    time_val, event_val, risk_scores_val,
                    time_train=time_train, event_train=event_train
                )
                
                for key in fold_metrics:
                    fold_metrics[key].append(metrics.get(key, np.nan))
                
                # Store first fold for KM plots
                if fold_idx == 1 and create_km_plots:
                    first_fold_data = {
                        'time_val': time_val,
                        'event_val': event_val,
                        'risk_val': risk_scores_val
                    }
            
            # Calculate summary statistics
            if fold_metrics['c_index']:
                results[result_key] = {
                    'config_name': config_name,
                    'model_type': model_type,
                    'c_index_mean': np.nanmean(fold_metrics['c_index']),
                    'c_index_std': np.nanstd(fold_metrics['c_index']),
                    'p_value_mean': np.nanmean(fold_metrics['p_value']),
                    'p_value_std': np.nanstd(fold_metrics['p_value']),
                    'hr_mean': np.nanmean(fold_metrics['hazard_ratio']),
                    'hr_std': np.nanstd(fold_metrics['hazard_ratio']),
                    'iauc_mean': np.nanmean(fold_metrics['iauc']),
                    'iauc_std': np.nanstd(fold_metrics['iauc']),
                    'iauc_min_mean': np.nanmean(fold_metrics['iauc_min']),
                    'iauc_max_mean': np.nanmean(fold_metrics['iauc_max']),
                    'n_folds': len([x for x in fold_metrics['c_index'] if not np.isnan(x)]),
                    'description': config['description']
                }
                
                print(f"      ✅ C-Index: {results[result_key]['c_index_mean']:.4f} ± {results[result_key]['c_index_std']:.4f}")
                print(f"         p-value: {results[result_key]['p_value_mean']:.4f} ± {results[result_key]['p_value_std']:.4f}")
                print(f"         HR:      {results[result_key]['hr_mean']:.4f} ± {results[result_key]['hr_std']:.4f}")
                
                if not np.isnan(results[result_key]['iauc_mean']):
                    print(f"         iAUC:    {results[result_key]['iauc_mean']:.4f} ± {results[result_key]['iauc_std']:.4f} " +
                          f"[{results[result_key]['iauc_min_mean']:.3f}-{results[result_key]['iauc_max_mean']:.3f}]")
                else:
                    print(f"         iAUC:    N/A (requires scikit-survival)")
                
                # Create KM plot
                if create_km_plots and first_fold_data is not None:
                    km_path = km_dir / f"{result_key}_validation.png"
                    plot_km_curve(
                        first_fold_data['time_val'],
                        first_fold_data['event_val'],
                        first_fold_data['risk_val'],
                        f"{config_name} ({model_type})",
                        km_path,
                        "Validation"
                    )
    
    return results


# =============================================================================
# SENSITIVITY ANALYSIS FOR DIGITAL TWIN PARAMETERS
# =============================================================================

def perform_sensitivity_analysis(df_merged: pd.DataFrame,
                                 radiomics_features: List[str],
                                 ae_features: List[str],
                                 clinical_features: List[str],
                                 dt_features: List[str],
                                 output_dir: Path,
                                 perturbation_levels: List[float] = [0.1, 0.2],
                                 random_state: int = 42) -> Dict:
    """
    Perform sensitivity analysis by perturbing α and β parameters.
    """
    
    print("\n" + "="*100)
    print("🔬 SENSITIVITY ANALYSIS: Perturbation of Digital Twin Parameters (α, β)")
    print("="*100)
    
    if len(dt_features) == 0:
        print("   ⚠️  No Digital Twin features available. Skipping sensitivity analysis.")
        return {}
    
    # Create output directory
    sens_dir = output_dir / "sensitivity_analysis"
    sens_dir.mkdir(parents=True, exist_ok=True)
    
    # Use stratified split to get training and test sets
    df_train, df_test = train_test_split(
        df_merged, 
        test_size=0.2, 
        random_state=random_state,
        stratify=df_merged['event'].astype(int)
    )
    
    print(f"\n📊 Dataset Split:")
    print(f"   • Training set: {len(df_train)} patients")
    print(f"   • Test set: {len(df_test)} patients (for sensitivity analysis)")
    
    # Identify DT parameter columns
    if 'Bio_Alpha' not in df_train.columns or 'Bio_Necrosis' not in df_train.columns:
        print("   ⚠️  Bio_Alpha or Bio_Necrosis not found. Cannot perform sensitivity analysis.")
        return {}
    
    # ========== STEP 1: Train baseline model ==========
    print(f"\n{'─'*100}")
    print("STEP 1: Training Baseline Model (Full DT Configuration)")
    print(f"{'─'*100}")
    
    # Select features
    ae_selected = select_features_univariate_cox(df_train, ae_features, TOP_K_AE)
    rad_selected = select_features_univariate_cox(df_train, radiomics_features, TOP_K_RADIOMICS)
    
    baseline_features = clinical_features + ae_selected + rad_selected + dt_features
    
    # Remove duplicates
    seen = set()
    baseline_features = [f for f in baseline_features if not (f in seen or seen.add(f))]
    
    print(f"   • Total features: {len(baseline_features)}")
    
    # Train baseline model
    scaler = StandardScaler()
    X_train = df_train[baseline_features].values
    X_test = df_test[baseline_features].values
    time_train = df_train['time_years'].values
    event_train = df_train['event'].values
    time_test = df_test['time_years'].values
    event_test = df_test['event'].values
    
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train CoxPH model
    df_train_scaled = pd.DataFrame(X_train_scaled, columns=baseline_features)
    df_train_scaled['time_years'] = time_train
    df_train_scaled['event'] = event_train
    
    df_test_scaled = pd.DataFrame(X_test_scaled, columns=baseline_features)
    
    baseline_cph = CoxPHFitter(penalizer=LASSO_PENALIZER)
    baseline_cph.fit(
        df_train_scaled[baseline_features + ['time_years', 'event']],
        duration_col='time_years',
        event_col='event'
    )
    
    # Get baseline predictions
    baseline_risk = baseline_cph.predict_partial_hazard(
        df_test_scaled[baseline_features]
    ).values.flatten()
    
    baseline_metrics = calculate_all_metrics(
        time_test, event_test, baseline_risk,
        time_train, event_train
    )
    
    print(f"\n   ✅ Baseline Model Performance:")
    print(f"      • C-Index: {baseline_metrics['c_index']:.4f}")
    print(f"      • p-value: {baseline_metrics['p_value']:.6f}")
    print(f"      • HR: {baseline_metrics['hazard_ratio']:.4f}")
    if not np.isnan(baseline_metrics['iauc']):
        print(f"      • iAUC: {baseline_metrics['iauc']:.4f}")
    
    # ========== STEP 2: Perform perturbations ==========
    print(f"\n{'─'*100}")
    print("STEP 2: Perturbing Parameters and Computing New Risk Scores")
    print(f"{'─'*100}")
    
    results = {
        'baseline_risk': baseline_risk,
        'baseline_metrics': baseline_metrics,
        'perturbations': {}
    }
    
    # Define perturbation scenarios
    scenarios = []
    for level in perturbation_levels:
        scenarios.extend([
            (f'alpha_plus_{int(level*100)}', 'Bio_Alpha', +level),
            (f'alpha_minus_{int(level*100)}', 'Bio_Alpha', -level),
            (f'beta_plus_{int(level*100)}', 'Bio_Necrosis', +level),
            (f'beta_minus_{int(level*100)}', 'Bio_Necrosis', -level),
            (f'both_plus_{int(level*100)}', 'both', +level),
            (f'both_minus_{int(level*100)}', 'both', -level),
        ])
    
    print(f"\n   📋 Perturbation Scenarios: {len(scenarios)}")
    
    for scenario_name, param_name, delta_fraction in tqdm(scenarios, desc="   Running scenarios"):
        
        # Create perturbed test set
        df_test_perturbed = df_test.copy()
        
        if param_name == 'Bio_Alpha':
            df_test_perturbed['Bio_Alpha'] = df_test['Bio_Alpha'] * (1 + delta_fraction)
            if 'Sim_Growth_Rate' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Growth_Rate'] = df_test['Sim_Growth_Rate'] * (1 + delta_fraction)
        
        elif param_name == 'Bio_Necrosis':
            df_test_perturbed['Bio_Necrosis'] = df_test['Bio_Necrosis'] * (1 + delta_fraction)
            if 'Sim_Necrosis_Ratio' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test['Sim_Necrosis_Ratio'] * (1 + delta_fraction)
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test_perturbed['Sim_Necrosis_Ratio'].clip(0, 1)
        
        elif param_name == 'both':
            df_test_perturbed['Bio_Alpha'] = df_test['Bio_Alpha'] * (1 + delta_fraction)
            df_test_perturbed['Bio_Necrosis'] = df_test['Bio_Necrosis'] * (1 + delta_fraction)
            if 'Sim_Growth_Rate' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Growth_Rate'] = df_test['Sim_Growth_Rate'] * (1 + delta_fraction)
            if 'Sim_Necrosis_Ratio' in df_test_perturbed.columns:
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test['Sim_Necrosis_Ratio'] * (1 + delta_fraction)
                df_test_perturbed['Sim_Necrosis_Ratio'] = df_test_perturbed['Sim_Necrosis_Ratio'].clip(0, 1)
        
        # Scale perturbed features
        X_test_perturbed = df_test_perturbed[baseline_features].values
        X_test_perturbed_scaled = scaler.transform(X_test_perturbed)
        df_test_perturbed_scaled = pd.DataFrame(X_test_perturbed_scaled, columns=baseline_features)
        
        # Predict with perturbed features
        perturbed_risk = baseline_cph.predict_partial_hazard(
            df_test_perturbed_scaled[baseline_features]
        ).values.flatten()
        
        # Compute metrics for perturbed predictions
        perturbed_metrics = calculate_all_metrics(
            time_test, event_test, perturbed_risk,
            time_train, event_train
        )
        
        # Analyze differences
        risk_diff = perturbed_risk - baseline_risk
        risk_diff_pct = (risk_diff / (baseline_risk + 1e-8)) * 100
        
        # Rank correlation
        spearman_rho, spearman_p = spearmanr(baseline_risk, perturbed_risk)
        kendall_tau, kendall_p = kendalltau(baseline_risk, perturbed_risk)
        
        # Risk category changes (stratified by median)
        baseline_median = np.median(baseline_risk)
        perturbed_median = np.median(perturbed_risk)
        
        baseline_high_risk = baseline_risk >= baseline_median
        perturbed_high_risk = perturbed_risk >= perturbed_median
        
        risk_category_changes = (baseline_high_risk != perturbed_high_risk).sum()
        pct_category_changes = 100 * risk_category_changes / len(baseline_risk)
        
        # Store results
        results['perturbations'][scenario_name] = {
            'param_name': param_name,
            'delta_fraction': delta_fraction,
            'delta_pct': delta_fraction * 100,
            'perturbed_risk': perturbed_risk,
            'risk_diff_mean': np.mean(risk_diff),
            'risk_diff_std': np.std(risk_diff),
            'risk_diff_abs_mean': np.mean(np.abs(risk_diff)),
            'risk_diff_pct_mean': np.mean(risk_diff_pct),
            'risk_diff_pct_abs_mean': np.mean(np.abs(risk_diff_pct)),
            'spearman_rho': spearman_rho,
            'spearman_p': spearman_p,
            'kendall_tau': kendall_tau,
            'kendall_p': kendall_p,
            'risk_category_changes': risk_category_changes,
            'pct_category_changes': pct_category_changes,
            'c_index': perturbed_metrics['c_index'],
            'c_index_diff': perturbed_metrics['c_index'] - baseline_metrics['c_index'],
            'p_value': perturbed_metrics['p_value'],
            'hazard_ratio': perturbed_metrics['hazard_ratio'],
            'iauc': perturbed_metrics['iauc']
        }
    
    # ========== STEP 3: Summarize results ==========
    print(f"\n{'─'*100}")
    print("STEP 3: Sensitivity Analysis Summary")
    print(f"{'─'*100}")
    
    # Create summary table
    summary_rows = []
    for scenario_name, scenario_data in results['perturbations'].items():
        summary_rows.append({
            'Scenario': scenario_name,
            'Parameter': scenario_data['param_name'],
            'Δ (%)': f"{scenario_data['delta_pct']:+.0f}%",
            'ΔRisk (abs mean)': scenario_data['risk_diff_abs_mean'],
            'ΔRisk (% mean)': scenario_data['risk_diff_pct_abs_mean'],
            'Spearman ρ': scenario_data['spearman_rho'],
            'Category Changes (%)': scenario_data['pct_category_changes'],
            'ΔC-Index': scenario_data['c_index_diff'],
            'C-Index': scenario_data['c_index']
        })
    
    df_summary = pd.DataFrame(summary_rows)
    
    # Save summary table
    summary_path = sens_dir / "sensitivity_summary.csv"
    df_summary.to_csv(summary_path, index=False)
    print(f"\n   ✅ Summary saved to: {summary_path}")
    
    # Print summary table
    print(f"\n   📊 SENSITIVITY ANALYSIS RESULTS:")
    print(f"   {'─'*98}")
    print(f"   {'Scenario':<22} {'Δ':<8} {'ΔRisk(abs)':<12} {'Spearman ρ':<12} {'Cat.Chg(%)':<12} {'ΔC-Idx':<10}")
    print(f"   {'─'*98}")
    
    for _, row in df_summary.iterrows():
        print(f"   {row['Scenario']:<22} {row['Δ (%)']:<8} "
              f"{row['ΔRisk (abs mean)']:>11.4f} {row['Spearman ρ']:>11.4f} "
              f"{row['Category Changes (%)']:>11.1f} {row['ΔC-Index']:>9.4f}")
    
    print(f"   {'─'*98}")
    
    # Save full results
    results_path = sens_dir / "sensitivity_analysis_full_results.pkl"
    with open(results_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"\n   💾 Full results saved to: {results_path}")
    
    return results


# =============================================================================
# RESULTS DISPLAY AND SAVING
# =============================================================================

def print_results_summary(results: Dict):
    """Print comprehensive results summary including iAUC and multiple models."""
    
    print("\n" + "="*100)
    print("📊 FINAL RESULTS SUMMARY WITH INTEGRATED AUC AND MULTIPLE MODELS")
    print("="*100)
    
    # Sort by C-Index
    sorted_results = sorted(results.items(), key=lambda x: x[1]['c_index_mean'], reverse=True)
    
    # Check if we have iAUC data
    has_iauc = any(not np.isnan(r[1]['iauc_mean']) for r in sorted_results)
    
    # Print results table
    header = "   ┌─────────────────────────────────────────┬──────────┬────────────────┬────────────┬────────────"
    if has_iauc:
        header += "┬────────────────┐"
    else:
        header += "┐"
    print(f"\n{header}")
    
    col_header = "   │ Configuration                           │ Model    │ C-Index        │ p-value    │ HR         "
    if has_iauc:
        col_header += "│ iAUC           │"
    else:
        col_header += "│"
    print(col_header)
    
    sep = "   ├─────────────────────────────────────────┼──────────┼────────────────┼────────────┼────────────"
    if has_iauc:
        sep += "┼────────────────┤"
    else:
        sep += "┤"
    print(sep)
    
    for result_key, data in sorted_results:
        config_name = data.get('config_name', result_key.rsplit('_', 1)[0])
        model_type = data.get('model_type', 'coxph')
        
        name_padded = config_name[:39].ljust(39)
        model_padded = model_type[:8].ljust(8)
        c_str = f"{data['c_index_mean']:.4f}±{data['c_index_std']:.4f}".center(14)
        p_str = f"{data['p_value_mean']:.4f}".center(10)
        hr_str = f"{data['hr_mean']:.2f}±{data['hr_std']:.2f}".center(10)
        
        if has_iauc:
            if not np.isnan(data['iauc_mean']):
                iauc_str = f"{data['iauc_mean']:.4f}±{data['iauc_std']:.4f}".center(14)
            else:
                iauc_str = "N/A".center(14)
            print(f"   │ {name_padded} │ {model_padded} │ {c_str} │ {p_str} │ {hr_str} │ {iauc_str} │")
        else:
            print(f"   │ {name_padded} │ {model_padded} │ {c_str} │ {p_str} │ {hr_str} │")
    
    footer = "   └─────────────────────────────────────────┴──────────┴────────────────┴────────────┴────────────"
    if has_iauc:
        footer += "┴────────────────┘"
    else:
        footer += "┘"
    print(footer)
    
    # Model comparison
    print("\n   📊 MODEL COMPARISON (averaged across configurations):")
    model_summary = {}
    for result_key, data in results.items():
        model_type = data.get('model_type', 'coxph')
        if model_type not in model_summary:
            model_summary[model_type] = {'c_indices': [], 'iaucs': []}
        model_summary[model_type]['c_indices'].append(data['c_index_mean'])
        if not np.isnan(data['iauc_mean']):
            model_summary[model_type]['iaucs'].append(data['iauc_mean'])
    
    for model_type, summary in sorted(model_summary.items(), 
                                       key=lambda x: np.mean(x[1]['c_indices']), 
                                       reverse=True):
        c_avg = np.mean(summary['c_indices'])
        c_std = np.std(summary['c_indices'])
        iauc_avg = np.mean(summary['iaucs']) if summary['iaucs'] else np.nan
        
        if not np.isnan(iauc_avg):
            print(f"      • {model_type.upper()}: Avg C-Index={c_avg:.4f}±{c_std:.4f}, Avg iAUC={iauc_avg:.4f}")
        else:
            print(f"      • {model_type.upper()}: Avg C-Index={c_avg:.4f}±{c_std:.4f}")
    
    # Highlight Digital Twin results
    dt_configs = [key for key in results.keys() if 'DT' in key]
    if dt_configs:
        print("\n   🔬 DIGITAL TWIN (3-D MLPA) RESULTS:")
        for result_key in sorted(dt_configs, key=lambda x: results[x]['c_index_mean'], reverse=True)[:5]:
            data = results[result_key]
            model = data.get('model_type', 'coxph')
            config = data.get('config_name', result_key)
            if has_iauc and not np.isnan(data['iauc_mean']):
                print(f"      • {config} ({model}): C={data['c_index_mean']:.4f}±{data['c_index_std']:.4f}, " +
                      f"HR={data['hr_mean']:.2f}, iAUC={data['iauc_mean']:.4f}")
            else:
                print(f"      • {config} ({model}): C={data['c_index_mean']:.4f}±{data['c_index_std']:.4f}, " +
                      f"HR={data['hr_mean']:.2f}")
    
    # Best configuration
    best_key = sorted_results[0][0]
    best_data = sorted_results[0][1]
    best_c = best_data['c_index_mean']
    best_model = best_data.get('model_type', 'coxph')
    best_config = best_data.get('config_name', best_key)
    
    print(f"\n   🏆 Best by C-Index: {best_config} ({best_model}) - C-Index: {best_c:.4f}")
    
    if has_iauc:
        sorted_by_iauc = sorted([(k, v) for k, v in results.items() if not np.isnan(v['iauc_mean'])], 
                                key=lambda x: x[1]['iauc_mean'], reverse=True)
        if sorted_by_iauc:
            best_iauc_key = sorted_by_iauc[0][0]
            best_iauc_data = sorted_by_iauc[0][1]
            best_iauc = best_iauc_data['iauc_mean']
            best_iauc_model = best_iauc_data.get('model_type', 'coxph')
            best_iauc_config = best_iauc_data.get('config_name', best_iauc_key)
            print(f"   🏆 Best by iAUC: {best_iauc_config} ({best_iauc_model}) - iAUC: {best_iauc:.4f}")


def save_results_to_csv(results: Dict, output_dir: Path):
    """Save results to CSV file including iAUC and model type."""
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    rows = []
    for result_key, data in results.items():
        config_name = data.get('config_name', result_key.rsplit('_', 1)[0])
        model_type = data.get('model_type', 'coxph')
        
        # Determine fusion type
        if 'Table5' in config_name:
            fusion_type = 'Table 5 (Feature Fusion)'
        elif 'Table6' in config_name:
            fusion_type = 'Table 6 (Signature Fusion)'
        else:
            fusion_type = 'Single Domain'
        
        # Check if DT is included
        has_dt = 'DT' in config_name
        
        rows.append({
            'Result_Key': result_key,
            'Configuration': config_name,
            'Model_Type': model_type,
            'Fusion_Type': fusion_type,
            'Includes_DT': has_dt,
            'C_Index_Mean': data['c_index_mean'],
            'C_Index_Std': data['c_index_std'],
            'P_Value_Mean': data['p_value_mean'],
            'P_Value_Std': data['p_value_std'],
            'HR_Mean': data['hr_mean'],
            'HR_Std': data['hr_std'],
            'iAUC_Mean': data['iauc_mean'],
            'iAUC_Std': data['iauc_std'],
            'iAUC_Min_Mean': data['iauc_min_mean'],
            'iAUC_Max_Mean': data['iauc_max_mean'],
            'N_Folds': data['n_folds'],
            'Description': data['description']
        })
    
    df = pd.DataFrame(rows).sort_values('C_Index_Mean', ascending=False)
    csv_path = output_dir / "results_with_3d_mlpa_iauc_multimodel.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n✅ Results saved to: {csv_path}")
    
    # Create pivot table by model
    pivot_path = output_dir / "results_pivot_by_model.csv"
    df_pivot = df.pivot_table(
        index='Configuration', 
        columns='Model_Type', 
        values=['C_Index_Mean', 'iAUC_Mean'],
        aggfunc='first'
    ).round(4)
    df_pivot.to_csv(pivot_path)
    print(f"✅ Pivot table saved to: {pivot_path}")
    
    # Also create a summary by iAUC
    if not df['iAUC_Mean'].isna().all():
        df_iauc = df.sort_values('iAUC_Mean', ascending=False)
        csv_iauc_path = output_dir / "results_sorted_by_iauc.csv"
        df_iauc.to_csv(csv_iauc_path, index=False)
        print(f"✅ iAUC-sorted results saved to: {csv_iauc_path}")
    
    return df


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main execution function."""
    
    start_time = time.time()
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "="*100)
    print("🚀 FERRETTI REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC + NEURAL COX + ENSEMBLE")
    print("="*100)
    print(f"\n📁 Output Directory: {OUTPUT_DIR}/")
    print(f"📊 Cross-Validation: {N_SPLITS}-fold")
    
    print("\n🔧 AVAILABLE MODELS:")
    print(f"   1. CoxPH (lifelines): ✅ Always available")
    if TORCH_AVAILABLE:
        print(f"   2. Neural Cox (PyTorch): ✅ Available")
        print(f"      • Hidden layers: {NEURAL_COX_HIDDEN_LAYERS}")
        print(f"      • Dropout: {NEURAL_COX_DROPOUT}")
        print(f"      • Epochs: {NEURAL_COX_EPOCHS}")
    else:
        print(f"   2. Neural Cox (PyTorch): ❌ Not available (pip install torch)")
    
    if SKSURV_AVAILABLE:
        print(f"   3. Gradient Boosting (scikit-survival): ✅ Available")
        print(f"   4. iAUC calculation: ✅ Available")
        print(f"      • Time points: {IAUC_TIME_POINTS.tolist()}")
    else:
        print(f"   3. Gradient Boosting: ❌ Not available (pip install scikit-survival)")
        print(f"   4. iAUC calculation: ❌ Not available")
    
    print(f"   5. Ensemble: {'✅ Available' if (TORCH_AVAILABLE or SKSURV_AVAILABLE) else '⚠️ CoxPH only'}")
    if TORCH_AVAILABLE or SKSURV_AVAILABLE:
        print(f"      • Weights: {ENSEMBLE_WEIGHTS}")
    
    print("\n" + "─"*100)
    print("🧬 3-D MLPA DIGITAL TWIN + MULTI-MODEL EXPLANATION:")
    print("─"*100)
    print("""
    The Digital Twin uses Multi-Level Parameterized Automata (MLPA) to simulate
    tumor growth based on radiomics-derived biological parameters:
    
    1. RADIOMICS → BIOLOGICAL PARAMETERS:
       • Entropy → α (proliferation): Higher entropy = more heterogeneous = faster growth
         Formula: α = 0.01 + 0.05 * normalized_entropy
       
       • Sphericity → β (necrosis): Lower sphericity = irregular shape = more necrosis
         Formula: β = 0.01 + 0.08 * (1 - sphericity)
    
    2. MODEL TYPES:
       • CoxPH: Classical linear proportional hazards model
       • Neural Cox (DeepSurv): Deep learning-based non-linear survival prediction
         - Uses Cox partial likelihood loss
         - Multiple hidden layers with dropout and batch normalization
         - Early stopping with validation set
       • Ensemble: Weighted combination of CoxPH, Neural Cox, and Gradient Boosting
         - Normalizes risk scores to [0, 1] range
         - Combines using configurable weights
    
    3. ALL CONFIGURATIONS TESTED WITH ALL MODELS:
       • Single domains (C, AE, R, DT)
       • Table 5 (Feature-level fusion)
       • Table 6 (Signature-level fusion) - NOW with Neural Cox & Ensemble!
    """)
    print("─"*100)
    
    # ========== Load All Data ==========
    df_clinical = load_clinical_data(CLINICAL_FILE)
    
    clinical_features = [
        'age', 'gender_encoded', 'T_Stage_encoded', 'N_Stage_encoded',
        'M_Stage_encoded', 'Overall_Stage_encoded', 'Histology_encoded'
    ]
    
    df_radiomics, radiomics_features, patient_ids_rad = load_radiomics_from_npy(
        RADIOMICS_NPY_FILE, RADIOMICS_NAMES_JSON, PATIENT_IDS_JSON
    )
    
    df_ae, ae_features = load_ae_features(DEEP_FEATURES_FILE)
    
    df_dt, dt_features = load_digital_twin_features(DT_FEATURES_FILE)
    
    # ========== Merge All Data ==========
    print("\n" + "="*100)
    print("📊 MERGING ALL DATA")
    print("="*100)
    
    df_clin_indexed = df_clinical.set_index('PatientID')
    df_rad_indexed = df_radiomics.set_index('PatientID')
    df_ae_indexed = df_ae.set_index('PatientID')
    
    df_merged = df_clin_indexed.join(df_rad_indexed, how='inner')
    df_merged = df_merged.join(df_ae_indexed, how='inner')
    
    if len(dt_features) > 0 and len(df_dt) > 0:
        df_dt_indexed = df_dt.set_index('PatientID')
        df_merged = df_merged.join(df_dt_indexed, how='inner')
        print(f"   ✅ Added Digital Twin features (inner join)")
    
    df_merged = df_merged.dropna(subset=['time_years', 'event'])
    
    print(f"\n✅ Final merged dataset: {len(df_merged)} patients")
    print(f"\n📊 Feature Summary:")
    print(f"   • Clinical:      {len(clinical_features)} features")
    print(f"   • Radiomics:     {len(radiomics_features)} features")
    print(f"   • AutoEncoder:   {len(ae_features)} features")
    print(f"   • Digital Twin:  {len(dt_features)} features")
    
    # Apply feature selection pipeline
    print("\n📊 Applying Ferretti Feature Selection Pipeline:")
    radiomics_selected = remove_correlated_features(df_merged, radiomics_features, CORRELATION_THRESHOLD)
    radiomics_selected = remove_constant_features(df_merged, radiomics_selected)
    print(f"   • Radiomics: {len(radiomics_features)} → {len(radiomics_selected)} features")
    
    if len(dt_features) > 0:
        dt_selected = remove_correlated_features(df_merged, dt_features, CORRELATION_THRESHOLD)
        dt_selected = remove_constant_features(df_merged, dt_selected)
        print(f"   • Digital Twin: {len(dt_features)} → {len(dt_selected)} features")
    else:
        dt_selected = []
    
    # ========== Run Experiment ==========
    results = run_experiment_with_3d_mlpa(
        df_merged=df_merged,
        radiomics_features=radiomics_selected,
        ae_features=ae_features,
        clinical_features=clinical_features,
        dt_features=dt_selected,
        n_splits=N_SPLITS,
        random_state=RANDOM_STATE,
        create_km_plots=CREATE_KM_PLOTS
    )
    
    # ========== Run Sensitivity Analysis ==========
    if RUN_SENSITIVITY_ANALYSIS and len(dt_selected) > 0:
        print("\n" + "="*100)
        print("🔬 RUNNING SENSITIVITY ANALYSIS")
        print("="*100)
        
        sensitivity_results = perform_sensitivity_analysis(
            df_merged=df_merged,
            radiomics_features=radiomics_selected,
            ae_features=ae_features,
            clinical_features=clinical_features,
            dt_features=dt_selected,
            output_dir=OUTPUT_DIR,
            perturbation_levels=SENSITIVITY_PERTURBATION_LEVELS,
            random_state=RANDOM_STATE
        )
    elif len(dt_selected) == 0:
        print("\n⚠️  Skipping sensitivity analysis (no Digital Twin features available)")
    
    # ========== Display & Save Results ==========
    print_results_summary(results)
    save_results_to_csv(results, OUTPUT_DIR)
    
    # Final summary
    end_time = time.time()
    duration = end_time - start_time
    
    print("\n" + "="*100)
    print("✅ ANALYSIS COMPLETE WITH NEURAL COX AND ENSEMBLE!")
    print("="*100)
    print(f"\n⏱️  Total Runtime: {duration/60:.1f} minutes")
    print(f"\n📁 Output Files:")
    print(f"   • Results CSV: {OUTPUT_DIR}/results_with_3d_mlpa_iauc_multimodel.csv")
    print(f"   • Pivot table: {OUTPUT_DIR}/results_pivot_by_model.csv")
    if SKSURV_AVAILABLE:
        print(f"   • iAUC-sorted: {OUTPUT_DIR}/results_sorted_by_iauc.csv")
    if CREATE_KM_PLOTS:
        print(f"   • KM Plots:    {OUTPUT_DIR}/km_plots/")
    if RUN_SENSITIVITY_ANALYSIS and len(dt_selected) > 0:
        print(f"   • Sensitivity: {OUTPUT_DIR}/sensitivity_analysis/")
    
    # Count results by model type
    model_counts = {}
    for result_key, data in results.items():
        model = data.get('model_type', 'coxph')
        model_counts[model] = model_counts.get(model, 0) + 1
    
    print("\n📊 Results by Model Type:")
    for model, count in sorted(model_counts.items()):
        print(f"   • {model.upper()}: {count} configurations")
    print(f"   • Total: {len(results)}")
    
    print("\n🔬 Key Findings:")
    
    # Best overall
    best = max(results.items(), key=lambda x: x[1]['c_index_mean'])
    print(f"   • Best overall: {best[1].get('config_name', best[0])} ({best[1].get('model_type', 'coxph')})")
    print(f"     C-Index = {best[1]['c_index_mean']:.4f}", end="")
    if not np.isnan(best[1]['iauc_mean']):
        print(f", iAUC = {best[1]['iauc_mean']:.4f}")
    else:
        print()
    
    # Best by model
    print("\n   • Best by model type:")
    for model in ['coxph', 'neural_cox', 'ensemble']:
        model_results = [(k, v) for k, v in results.items() if v.get('model_type', 'coxph') == model]
        if model_results:
            best_model = max(model_results, key=lambda x: x[1]['c_index_mean'])
            print(f"     - {model.upper()}: {best_model[1].get('config_name', best_model[0])} " +
                  f"(C={best_model[1]['c_index_mean']:.4f})")
    
    if SKSURV_AVAILABLE:
        print("\n📈 iAUC Interpretation:")
        print("   • iAUC > 0.70: Excellent time-dependent discrimination")
        print("   • iAUC 0.60-0.70: Good discrimination")
        print("   • iAUC 0.50-0.60: Moderate discrimination")
        print("   • iAUC = 0.50: Random prediction")
    
    if not TORCH_AVAILABLE:
        print("\n⚠️  To enable Neural Cox:")
        print("   pip install torch")
    
    if not SKSURV_AVAILABLE:
        print("\n⚠️  To enable iAUC and Gradient Boosting:")
        print("   pip install scikit-survival")
    
    print("\n" + "="*100)


if __name__ == "__main__":
    main()


🚀 FERRETTI REPLICATION WITH 3-D MLPA DIGITAL TWIN + iAUC + NEURAL COX + ENSEMBLE

📁 Output Directory: ferretti_with_3d_mlpa_iauc_output/
📊 Cross-Validation: 5-fold

🔧 AVAILABLE MODELS:
   1. CoxPH (lifelines): ✅ Always available
   2. Neural Cox (PyTorch): ✅ Available
      • Hidden layers: [64, 32]
      • Dropout: 0.3
      • Epochs: 100
   3. Gradient Boosting (scikit-survival): ✅ Available
   4. iAUC calculation: ✅ Available
      • Time points: [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0]
   5. Ensemble: ✅ Available
      • Weights: {'coxph': 0.4, 'neural_cox': 0.3, 'gradient_boosting': 0.3}

────────────────────────────────────────────────────────────────────────────────────────────────────
🧬 3-D MLPA DIGITAL TWIN + MULTI-MODEL EXPLANATION:
────────────────────────────────────────────────────────────────────────────────────────────────────

    The Digital Twin uses Multi-Level Parameterized Automata (MLPA) to simulate
    tumor growth based on radiomics-derived biological param


   📈 Model: NEURAL_COX
      ✅ C-Index: 0.5534 ± 0.0353
         p-value: 0.2317 ± 0.2680
         HR:      1.3468 ± 0.4898
         iAUC:    0.5754 ± 0.0525 [0.529-0.626]

   📈 Model: ENSEMBLE
      ✅ C-Index: 0.5485 ± 0.0391
         p-value: 0.4154 ± 0.3864
         HR:      1.1212 ± 0.3100
         iAUC:    0.5772 ± 0.0490 [0.534-0.639]

────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: Table5_AE_R
   📊 TABLE 5: AE + Radiomics

   📈 Model: COXPH
      ✅ C-Index: 0.5884 ± 0.0327
         p-value: 0.2255 ± 0.3231
         HR:      1.5141 ± 0.3278
         iAUC:    0.6458 ± 0.0520 [0.595-0.697]

   📈 Model: NEURAL_COX
      ✅ C-Index: 0.5906 ± 0.0450
         p-value: 0.2936 ± 0.3380
         HR:      1.4717 ± 0.3557
         iAUC:    0.6428 ± 0.0690 [0.576-0.699]

   📈 Model: ENSEMBLE
      ✅ C-Index: 0.5712 ± 0.0264
         p-value: 0.3129 ± 0.3617
         HR:      1.4789 ± 0.4720
         iAUC:    0.6203 ± 0.031


   📈 Model: NEURAL_COX
      ✅ C-Index: 0.5991 ± 0.0157
         p-value: 0.0367 ± 0.0436
         HR:      1.7505 ± 0.1711
         iAUC:    0.6490 ± 0.0353 [0.598-0.712]

   📈 Model: ENSEMBLE
      ✅ C-Index: 0.6066 ± 0.0092
         p-value: 0.0522 ± 0.0398
         HR:      1.6856 ± 0.2109
         iAUC:    0.6553 ± 0.0336 [0.593-0.716]

────────────────────────────────────────────────────────────────────────────────────────────────────
🔬 Configuration: Table6_DT_C
   ⭐ TABLE 6: DT + Clinical signatures

   📈 Model: COXPH
      ✅ C-Index: 0.5951 ± 0.0408
         p-value: 0.1842 ± 0.3455
         HR:      1.7226 ± 0.4022
         iAUC:    0.6443 ± 0.0404 [0.576-0.740]

   📈 Model: NEURAL_COX
      ✅ C-Index: 0.5881 ± 0.0321
         p-value: 0.2041 ± 0.2295
         HR:      1.6154 ± 0.4764
         iAUC:    0.6363 ± 0.0259 [0.570-0.726]

   📈 Model: ENSEMBLE
      ✅ C-Index: 0.6009 ± 0.0399
         p-value: 0.1976 ± 0.3229
         HR:      1.8227 ± 0.5037
         iAUC:    0.65

   Running scenarios: 100%|██████████| 12/12 [00:00<00:00, 17.73it/s]


────────────────────────────────────────────────────────────────────────────────────────────────────
STEP 3: Sensitivity Analysis Summary
────────────────────────────────────────────────────────────────────────────────────────────────────

   ✅ Summary saved to: ferretti_with_3d_mlpa_iauc_output/sensitivity_analysis/sensitivity_summary.csv

   📊 SENSITIVITY ANALYSIS RESULTS:
   ──────────────────────────────────────────────────────────────────────────────────────────────────
   Scenario               Δ        ΔRisk(abs)   Spearman ρ   Cat.Chg(%)   ΔC-Idx    
   ──────────────────────────────────────────────────────────────────────────────────────────────────
   alpha_plus_10          +10%          0.0112      0.9996         0.0    0.0003
   alpha_minus_10         -10%          0.0110      0.9995         0.0    0.0031
   beta_plus_10           +10%          0.0361      0.9993         0.0   -0.0010
   beta_minus_10          -10%          0.0349      0.9994         0.0    0.0000
   both_